<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>Artificial Intelligence and Machine Learning</center></font>
<center><font size=6>Natural Language Processing with Generative AI - Project</center></font>

<center><p float="center">
  <img src="https://i.ibb.co/Q325rK84/medical.png" width="480"/>
</p></center>

<center><font size=6>Medical Assistant</center></font>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## **Please read the instructions carefully before starting the project.**

This is a blank Python Notebook file in which only the section headers, task descriptions, and code comments are provided. Unlike the low-code version, no starter code or blanks ('_____') are given here.
* Read the comment(s) above each empty code cell carefully to understand the task, and then write the complete code yourself in that cell.
* Please run the codes in a sequential manner from the beginning to avoid any unnecessary errors.
* Add the results/observations (wherever mentioned) derived from the analysis in the presentation and submit the same.

**Note**: To run the LLM calls in this notebook, you will need an OpenAI-compatible API key (accessed here via the GL Proxy).

- Create a `config.json` file in the same directory as this notebook (e.g. `/content/config.json` on Google Colab) with the following structure:
```
{"OPENAI_API_KEY": "sk-...", "OPENAI_API_BASE": "https://.../v1"}
```
- Upload this `config.json` file to your working directory (e.g. via the Colab file browser) before running the "Setting up the OpenAI Client" cell below.
- Do **not** share your `config.json` file or commit it to any public repository, since it contains your private API key.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# ==============================================================
# SETUP-A: INSTALL REQUIRED LIBRARIES
# Medical Assistant (RAG) - Merck Manual
# ==============================================================

# -q = quiet mode. Hides the long install log so the exported HTML stays clean.
# Versions are pinned (==) so the code behaves exactly like the course template
# was tested. Unpinned libraries (especially langchain) change fast and break code.
# NOTE: no comments are allowed on the lines below, because the "\" at the end
# of each line joins them into ONE command. A comment would break that chain.
%pip install -q \
    pandas==2.2.2 \
    tiktoken==0.12.0 \
    pymupdf==1.26.5 \
    langchain==0.3.27 \
    langchain-community==0.3.31 \
    langchain-huggingface==0.3.1 \
    chromadb==1.1.1 \
    sentence-transformers \
    openai==2.50.0

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [17]:
# ==============================================================
# SETUP-B: LIBRARY IMPORTS
# Medical Assistant (RAG) - Merck Manual
# ==============================================================

# --- Suppress all warnings FIRST, before anything else ---
# LangChain prints "deprecation" warnings on import (pink boxes in the notebook).
# This line must run BEFORE the langchain imports, or the warnings still appear.
import warnings
warnings.filterwarnings("ignore")

# --- Tell Hugging Face to ignore TensorFlow, use PyTorch only ---
# TensorFlow is installed on this machine but is broken by the protobuf upgrade in SETUP-A.
# Without these lines, sentence-transformers may try to load TensorFlow and crash.
# They must run BEFORE any langchain / huggingface import to take effect.
import os                                # os: environment settings + file/folder handling (e.g. vector DB folder)
os.environ["USE_TF"] = "0"               # transformers library: skip the TensorFlow backend
os.environ["TRANSFORMERS_NO_TF"] = "1"   # older flag name for the same thing, set both to be safe
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # hide download/loading progress bars (clean HTML export)
os.environ["HF_HUB_VERBOSITY"] = "error"           # hide the "unauthenticated requests" notice
os.environ["TRANSFORMERS_VERBOSITY"] = "error"     # show only real errors from the transformers library

# --- Python standard libraries ---
import json                  # json: read the API key + base URL from config.json
import re                    # re: pattern matching, used to strip dotted leader lines (.....) from the text

# --- Data handling ---
import pandas as pd          # pandas: tabulate evaluation scores at the end for a clean summary
import tiktoken              # tiktoken: counts TOKENS (not characters) so chunk sizes match how the LLM reads text

# --- Hardware check ---
import torch                 # torch: tells us if a GPU is available (embeddings run much faster on GPU)

# --- Step 1 of RAG: Load the PDF ---
from langchain_community.document_loaders import PyMuPDFLoader
                             # PyMuPDFLoader: reads the 4,000+ page Merck Manual, one Document per page

# --- Step 2 of RAG: Split into chunks ---
from langchain.text_splitter import RecursiveCharacterTextSplitter
                             # splits long pages into smaller overlapping chunks the model can search

# --- Step 3 of RAG: Convert text to numbers (embeddings) ---
from langchain_huggingface import HuggingFaceEmbeddings
                             # turns each chunk into a vector so "similar meaning" = "close numbers"

# --- Step 4 of RAG: Store and search the vectors ---
from langchain_community.vectorstores import Chroma
                             # Chroma: vector database that stores embeddings and finds the closest chunks

# --- The LLM (answer generator + judge) ---
from openai import OpenAI    # OpenAI client: sends prompts to the model through the GL Proxy

# --- Display settings ---
from IPython.display import Markdown, display
                             # renders LLM answers as formatted text (headings, bold) instead of a raw string

pd.set_option("display.max_colwidth", None)   # show full text in DataFrame cells, no "..." truncation

# --- Confirm setup ---
# Quick sanity check so you know everything loaded and which hardware you're on.
print("All libraries imported successfully!")
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

All libraries imported successfully!
PyTorch version: 2.6.0+cu124
GPU available: True


## Question Answering using LLM

### Setting up the OpenAI Client

In [21]:
# ==============================================================
# CELL 1: SET UP THE OPENAI CLIENT
# ==============================================================

# --- Locate config.json ---
# Local Jupyter runs from the notebook's own folder, so the plain file name works.
# The template's "/content/config.json" path only exists on Google Colab.
config_path = "config.json"

# --- Read the credentials from the file ---
# Keeping the key in a separate file (not in the code) means it never
# appears in the exported HTML you submit.
with open(config_path, "r") as f:
    config = json.load(f)            # converts the JSON text into a Python dictionary

# --- Get the API key ---
api_key = config["OPENAI_API_KEY"]

# --- Get the base URL (accept both key names) ---
# The notebook uses "OPENAI_API_BASE" but the course page says "OPENAI_BASE_URL".
# Checking for both avoids a KeyError whichever version you used.
if "OPENAI_API_BASE" in config:
    base_url = config["OPENAI_API_BASE"]
elif "OPENAI_BASE_URL" in config:
    base_url = config["OPENAI_BASE_URL"]
else:
    raise KeyError("config.json must contain OPENAI_API_BASE or OPENAI_BASE_URL")

# --- Create the client ---
# The client is our "phone line" to the model: every question we send goes through it.
client = OpenAI(
    api_key=api_key,       # proves who we are
    base_url=base_url      # sends requests to the GL Proxy instead of OpenAI directly
)

# --- Confirm setup (without exposing secrets) ---
# Never print the actual key: the output ends up in the HTML you submit.
print("OpenAI client created successfully!")
print("API key loaded:", len(api_key) > 0)
print("Base URL loaded:", len(base_url) > 0)

OpenAI client created successfully!
API key loaded: True
Base URL loaded: True


In [23]:
# Separate models are used for response generation and for LLM-as-a-judge evaluation.
# Using a different (and typically stronger) model as the judge avoids the generator
# grading its own homework, which is a known bias in LLM-as-a-judge setups.
# Write code to define two variables:
#   llm        -> the model used for generating answers (base LLM, prompt-engineered LLM, and RAG)
#   judge_llm  -> the model used only for the groundedness/relevance judge in the evaluation section

# ==============================================================
# CELL 2: DEFINE THE GENERATOR AND JUDGE MODELS
# ==============================================================

# --- Generator model: answers every question ---
# gpt-4o-mini is fast, cheap and strong enough for question answering.
# Cheap matters here: our key has a $4/day limit, and we'll make 30+ calls.
llm = "gpt-4o-mini"

# --- Judge model: grades the answers in the evaluation section ---
# A DIFFERENT, stronger model avoids "self-preference bias":
# a model tends to rate its own writing too kindly.
judge_llm = "gpt-4o"

# --- Quick check that both models are available through the GL Proxy ---
# One tiny call per model; max_tokens=5 keeps the cost near zero.
for model_name in [llm, judge_llm]:
    try:
        test = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": "Reply with the single word: OK"}],
            max_tokens=5,
        )
        print(f"{model_name:<12} -> available (replied: {test.choices[0].message.content})")
    except Exception as error:
        print(f"{model_name:<12} -> NOT available: {error}")

gpt-4o-mini  -> available (replied: OK)
gpt-4o       -> available (replied: OK)


### Response

In [24]:
# Write a function called response() that:
#   - takes a query string and generation parameters (max_tokens, temperature, top_p) as input
#   - calls the OpenAI client (client.chat.completions.create) using the llm model
#   - returns only the generated text of the response

# ==============================================================
# CELL 3: THE response() FUNCTION
# Sends one question to the LLM and returns only the answer text
# ==============================================================

def response(query, max_tokens=512, temperature=0, top_p=0.95):
    """
    Send a query to the generator model (llm) and return the answer text.

    Parameters
    ----------
    query       : the question (or prompt) to send
    max_tokens  : upper limit on answer length (~512 tokens = ~380 words)
    temperature : randomness. 0 = focused and repeatable, higher = more creative
    top_p       : "nucleus sampling". Only consider the most likely words that
                  together make up this share of probability (0.95 = top 95%)
    """

    # --- Call the model ---
    # messages is a list of chat turns; here it's a single user message.
    completion = client.chat.completions.create(
        model=llm,                       # gpt-4o-mini, defined in CELL 2
        messages=[{"role": "user", "content": query}],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
    )

    # --- Pull out just the answer text ---
    # The API returns a large object; the text lives in choices[0].message.content.
    answer = completion.choices[0].message.content
    return answer


print("response() function is ready.")

response() function is ready.


In [25]:
# Call the response() function with the query: "What treatment options are available for managing hypertension?"

# ==============================================================
# CELL 4: TEST response() WITH A SAMPLE QUESTION
# ==============================================================

# --- Define the question ---
test_query = "What treatment options are available for managing hypertension?"

# --- Ask the LLM (no manual, no special instructions, default settings) ---
# This is our "baseline": what the model knows from its training data alone.
test_answer = response(test_query)

# --- Show the answer nicely formatted ---
# Markdown() renders headings, bullet points and bold text
# instead of showing raw symbols like ** and ###.
display(Markdown(test_answer))

Managing hypertension (high blood pressure) typically involves a combination of lifestyle changes and, if necessary, medication. Here are the main treatment options:

### Lifestyle Modifications
1. **Dietary Changes**:
   - **DASH Diet**: Emphasizes fruits, vegetables, whole grains, and low-fat dairy while reducing saturated fat and cholesterol.
   - **Reduce Sodium Intake**: Aim for less than 2,300 mg per day, or ideally 1,500 mg for greater blood pressure reduction.
   - **Limit Alcohol**: Moderate consumption is advised (up to one drink per day for women and two for men).

2. **Physical Activity**:
   - Engage in regular aerobic exercise (e.g., walking, jogging, cycling) for at least 150 minutes per week.

3. **Weight Management**:
   - Achieving and maintaining a healthy weight can significantly lower blood pressure.

4. **Stress Management**:
   - Techniques such as meditation, deep breathing exercises, and yoga can help reduce stress levels.

5. **Quit Smoking**:
   - Smoking cessation improves overall cardiovascular health and can help lower blood pressure.

### Medications
If lifestyle changes alone are insufficient to control blood pressure, healthcare providers may prescribe medications. Common classes of antihypertensive drugs include:

1. **Diuretics**: Help the body eliminate excess sodium and water, reducing blood volume.
   - Examples: Hydrochlorothiazide, Chlorthalidone.

2. **ACE Inhibitors**: Help relax blood vessels by inhibiting the formation of a hormone that narrows blood vessels.
   - Examples: Lisinopril, Enalapril.

3. **Angiotensin II Receptor Blockers (ARBs)**: Block the action of the hormone that narrows blood vessels.
   - Examples: Losartan, Valsartan.

4. **Calcium Channel Blockers**: Prevent calcium from entering heart and blood vessel cells, leading to relaxed blood vessels.
   - Examples: Amlodipine, Diltiazem.

5. **Beta-Blockers**: Reduce heart rate and the heart's workload.
   - Examples: Metoprolol, Atenolol.

6. **Renin Inhibitors**: Directly inhibit renin, an enzyme involved in blood pressure regulation.
   - Example: Aliskiren.

7. **Alpha-Blockers**: Help reduce nerve signals to blood vessels, allowing them to relax.
   - Example: Doxazos

**Observations:**

- The base LLM (`gpt-4o-mini`, default settings, no instructions) gave a well-structured answer split into **lifestyle modifications** (DASH diet, sodium < 2,300 mg/day, 150 min/week of exercise, weight and stress management, quitting smoking) and **seven medication classes** (diuretics, ACE inhibitors, ARBs, calcium channel blockers, beta-blockers, renin inhibitors, alpha-blockers) with example drug names.
- The content is clinically reasonable and matches standard hypertension guidance, showing strong general medical knowledge from training data.
- The answer was **cut off mid-word** ("Example: Doxazos…"), so `max_tokens=512` is too low for detailed medical answers and this parameter needs tuning.
- It **cites no source**: a clinician can't verify where the information came from, which is a key limitation for medical use.
- It is generic: no treatment thresholds, no first-line choice by patient group, and no monitoring guidance. Prompt engineering and RAG (grounding in the Merck Manual) aim to close this gap.


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [26]:
# Write code to define the query above as user_input_1 and pass it to the response() function

# ==============================================================
# CELL 5: QUERY 1 - SEPSIS (base LLM, no instructions)
# ==============================================================

# --- Define the question exactly as given in the problem statement ---
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"

# --- Ask the base LLM ---
# max_tokens raised from 512 to 800 because the hypertension test was cut off
# mid-word. Everything else stays at the defaults (temperature=0, top_p=0.95).
answer_1 = response(user_input_1, max_tokens=800)

# --- Show the formatted answer ---
display(Markdown(answer_1))

Managing sepsis in a critical care unit involves a systematic approach that includes early recognition, prompt intervention, and ongoing monitoring. The following steps outline the general protocol for managing sepsis:

### 1. **Early Recognition**
   - **Identify Symptoms**: Look for signs of sepsis, which may include fever, hypothermia, tachycardia, tachypnea, altered mental status, and hypotension.
   - **Use Screening Tools**: Utilize tools like the Sequential Organ Failure Assessment (SOFA) score or the quick SOFA (qSOFA) to assess the severity of the patient's condition.

### 2. **Initial Assessment**
   - **Obtain a Detailed History**: Gather information on the patient's medical history, recent infections, and any risk factors for sepsis.
   - **Physical Examination**: Conduct a thorough physical exam to identify potential sources of infection.

### 3. **Immediate Interventions**
   - **Administer Oxygen**: Ensure adequate oxygenation and consider supplemental oxygen if the patient is hypoxic.
   - **Fluid Resuscitation**: Administer intravenous (IV) fluids (usually crystalloids) rapidly, typically 30 mL/kg within the first 3 hours, to restore hemodynamic stability.
   - **Vasopressors**: If hypotension persists despite adequate fluid resuscitation, initiate vasopressors (e.g., norepinephrine) to maintain mean arterial pressure (MAP) ≥ 65 mmHg.

### 4. **Source Control**
   - **Identify and Treat the Infection**: Obtain cultures (blood, urine, sputum, etc.) before starting antibiotics, if possible. Initiate broad-spectrum IV antibiotics within the first hour of recognition of sepsis.
   - **Consider Surgical Intervention**: If there is a clear source of infection (e.g., abscess, infected tissue), consult surgical services for potential drainage or removal.

### 5. **Monitoring and Supportive Care**
   - **Continuous Monitoring**: Monitor vital signs, urine output, and laboratory values (e.g., lactate levels, complete blood count, renal function).
   - **Organ Support**: Provide supportive care for affected organ systems, which may include renal replacement therapy for acute kidney injury, mechanical ventilation for respiratory failure, and other interventions as needed.

### 6. **Reassessment**
   - **Reevaluate Response to Treatment**: After initial interventions, reassess the patient’s hemodynamic status, organ function, and clinical signs of infection.
   - **Adjust Treatment as Necessary**: Modify fluid resuscitation, vasopressor therapy, and antibiotic regimens based on the patient's response and culture results.

### 7. **Long-term Management**
   - **Transition to Appropriate Care**: Once stabilized, consider transferring the patient to a step-down unit or regular ward as appropriate.
   - **Prevent Complications**: Implement strategies to prevent complications associated with sepsis and critical illness, such as deep vein thrombosis (DVT) prophylaxis and stress ulcer prophylaxis.

### 8. **Education and Follow-up**
   - **Educate the Patient and Family**: Provide information about sepsis, its implications, and the recovery process.
   - **Plan for Follow-up**: Arrange for follow-up care to monitor recovery and address any long-term effects of sepsis.

### Conclusion
The management of sepsis in a critical care unit is a dynamic process that requires a multidisciplinary approach, timely interventions, and continuous monitoring to improve patient outcomes. Adhering to established guidelines, such as those from the Surviving Sepsis Campaign, can help ensure best practices are followed.

**Observations:**
- With `max_tokens` raised to 800, the answer **completed in full** (ending with a conclusion), confirming the cut-off in the hypertension test was caused by the 512-token limit.
- The response is well structured into 8 phases: early recognition (SOFA/qSOFA), initial assessment, immediate interventions, source control, monitoring, reassessment, long-term management and follow-up.
- It includes several correct, specific clinical targets: **30 mL/kg IV crystalloids within 3 hours**, **norepinephrine to keep MAP ≥ 65 mmHg**, cultures before antibiotics, and **broad-spectrum antibiotics within the first hour**.
- Some important items are missing compared with the Merck Manual chunk retrieved earlier (page 2456): **corticosteroids** for refractory septic shock and **blood glucose control** aren't mentioned, and no specific antibiotic choices are given.
- It references the Surviving Sepsis Campaign but gives **no verifiable source or page**; the content comes from general training data, not the manual.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [27]:
# Write code to define the query above as user_input_2 and pass it to the response() function

# ==============================================================
# CELL 6: QUERY 2 - APPENDICITIS (base LLM, no instructions)
# ==============================================================

# --- Define the question exactly as given in the problem statement ---
user_input_2 = ("What are the common symptoms for appendicitis, and can it be cured via medicine? "
                "If not, what surgical procedure should be followed to treat it?")

# --- Ask the base LLM (same settings as Query 1 for a fair comparison) ---
answer_2 = response(user_input_2, max_tokens=800)

# --- Show the formatted answer ---
display(Markdown(answer_2))

Common symptoms of appendicitis include:

1. **Abdominal Pain**: Typically starts around the navel and then moves to the lower right abdomen.
2. **Loss of Appetite**: A sudden decrease in appetite is common.
3. **Nausea and Vomiting**: Often follows the onset of abdominal pain.
4. **Fever**: A low-grade fever may develop.
5. **Constipation or Diarrhea**: Changes in bowel habits can occur.
6. **Abdominal Swelling**: The abdomen may become swollen or tender.

Appendicitis cannot be effectively treated with medication alone. The standard treatment is surgical removal of the appendix, known as an **appendectomy**. This can be performed using two main methods:

1. **Open Appendectomy**: A larger incision is made in the lower right abdomen to remove the appendix.
2. **Laparoscopic Appendectomy**: This is a minimally invasive procedure where several small incisions are made, and the appendix is removed with the aid of a camera and special instruments.

Laparoscopic appendectomy is often preferred due to its benefits, including less postoperative pain, shorter recovery time, and minimal scarring. However, the choice of procedure may depend on the patient's specific situation and the surgeon's expertise. 

If you suspect appendicitis, it is important to seek medical attention promptly, as a delayed diagnosis can lead to complications such as a ruptured appendix.

**Observations:**
- The answer covers all three parts of the question: **symptoms** (periumbilical pain migrating to the lower right abdomen, loss of appetite, nausea/vomiting, low-grade fever, bowel changes, tenderness), **whether medicine can cure it**, and the **surgical procedure** (open vs laparoscopic appendectomy).
- It correctly identifies **appendectomy** as the standard treatment and explains why laparoscopic surgery is often preferred (less pain, faster recovery, less scarring).
- It **oversimplifies** by stating appendicitis "cannot be effectively treated with medication alone": antibiotic-only treatment is a recognised option for selected patients with uncomplicated appendicitis, a nuance the base model misses.
- Key clinical details are absent: no **diagnostic workup** (CT/ultrasound, labs), no **pre-operative antibiotics**, and no management of a perforated appendix or abscess (e.g., drainage).
- The answer was short and finished well within the 800-token limit; like Query 1, it cites **no source**.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [28]:
# Write code to define the query above as user_input_3 and pass it to the response() function

# ==============================================================
# CELL 7: QUERY 3 - SUDDEN PATCHY HAIR LOSS (base LLM, no instructions)
# ==============================================================

# --- Define the question exactly as given in the problem statement ---
user_input_3 = ("What are the effective treatments or solutions for addressing sudden patchy hair loss, "
                "commonly seen as localized bald spots on the scalp, and what could be the possible "
                "causes behind it?")

# --- Ask the base LLM (same settings as Queries 1-2 for a fair comparison) ---
answer_3 = response(user_input_3, max_tokens=800)

# --- Show the formatted answer ---
display(Markdown(answer_3))

Sudden patchy hair loss, often referred to as alopecia areata, can manifest as localized bald spots on the scalp or other areas of the body. Here are some effective treatments and solutions, as well as potential causes:

### Possible Causes:
1. **Autoimmune Disorders**: The immune system mistakenly attacks hair follicles, leading to hair loss.
2. **Genetics**: A family history of alopecia or other autoimmune conditions can increase risk.
3. **Stress**: Physical or emotional stress can trigger hair loss in some individuals.
4. **Hormonal Changes**: Changes in hormones, such as those occurring during pregnancy or menopause, can contribute.
5. **Nutritional Deficiencies**: Lack of essential nutrients, such as iron, zinc, or vitamin D, may play a role.
6. **Infections**: Fungal infections like tinea capitis can cause patchy hair loss.
7. **Other Medical Conditions**: Conditions like thyroid disease or vitiligo can also lead to hair loss.

### Effective Treatments:
1. **Topical Corticosteroids**: These are often the first line of treatment to reduce inflammation and suppress the immune response.
2. **Minoxidil (Rogaine)**: This over-the-counter topical treatment can stimulate hair growth and is sometimes used in conjunction with other therapies.
3. **Intralesional Corticosteroid Injections**: For more severe cases, injections directly into the bald spots can promote hair regrowth.
4. **Immunotherapy**: Treatments like diphencyprone (DPCP) can provoke an allergic reaction that may help hair regrow.
5. **Oral Corticosteroids**: In some cases, systemic corticosteroids may be prescribed for more extensive hair loss.
6. **Anthralin**: A topical medication that can help stimulate hair growth by irritating the skin.
7. **Platelet-Rich Plasma (PRP) Therapy**: This involves injecting components derived from the patient’s own blood into the scalp to promote hair growth.
8. **Lifestyle Changes**: Managing stress through techniques like yoga, meditation, or counseling can be beneficial.
9. **Nutritional Support**: Ensuring a balanced diet rich in vitamins and minerals can support overall hair health.

### Additional Considerations:
- **Consult a Dermatologist**: It’s essential to get a proper diagnosis and treatment plan tailored to the individual’s specific condition.
- **Support Groups**: Joining support groups can help individuals cope with the emotional aspects of hair loss.
- **Wigs and Hairpieces**: For those experiencing significant hair loss, wigs or hairpieces can provide a cosmetic solution.

### Conclusion:
Addressing sudden patchy hair loss involves understanding the underlying causes and exploring various treatment options. Early intervention can often lead to better outcomes, so seeking professional advice is crucial.

**Observations:**
- The model correctly identifies the condition as **alopecia areata** and lists a broad set of treatments: topical, intralesional and oral **corticosteroids**, **minoxidil**, **immunotherapy (DPCP)**, **anthralin**, PRP, plus lifestyle and nutritional support.
- The "causes" section **mixes the true cause with look-alikes**: alopecia areata is autoimmune, while tinea capitis (fungal) is a different diagnosis, and hormonal changes such as pregnancy or menopause usually cause *diffuse* shedding, not patchy bald spots. A clinician would expect these separated into cause vs. differential diagnosis.
- Key look-alikes and diagnostic steps are missing: no mention of **trichotillomania** or **traction alopecia**, and no **workup** (hair pull test, dermoscopy, KOH/fungal test for tinea).
- Treatments are listed without clear ordering by severity or evidence strength; for example, PRP has weaker evidence than intralesional steroids but is presented alongside them.
- The answer completed within the 800-token limit, sensibly advises seeing a dermatologist, and cites **no source**.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [29]:
# Write code to define the query above as user_input_4 and pass it to the response() function

# ==============================================================
# CELL 8: QUERY 4 - TRAUMATIC BRAIN INJURY (base LLM, no instructions)
# ==============================================================

# --- Define the question exactly as given in the problem statement ---
user_input_4 = ("What treatments are recommended for a person who has sustained a physical injury "
                "to brain tissue, resulting in temporary or permanent impairment of brain function?")

# --- Ask the base LLM (same settings as Queries 1-3 for a fair comparison) ---
answer_4 = response(user_input_4, max_tokens=800)

# --- Show the formatted answer ---
display(Markdown(answer_4))

The treatment for a person who has sustained a physical injury to brain tissue, such as a traumatic brain injury (TBI), can vary widely depending on the severity of the injury, the specific areas of the brain affected, and the resulting impairments. Here are some common approaches to treatment:

1. **Emergency Care**: 
   - Immediate medical attention is crucial. This may involve stabilizing the patient, monitoring vital signs, and performing imaging studies (like CT or MRI scans) to assess the extent of the injury.

2. **Surgical Interventions**: 
   - In some cases, surgery may be necessary to relieve pressure on the brain, remove blood clots (hematomas), or repair skull fractures.

3. **Medication**: 
   - Medications may be prescribed to manage symptoms such as pain, seizures, or inflammation. Corticosteroids may be used to reduce swelling in the brain.

4. **Rehabilitation**: 
   - **Physical Therapy**: To improve mobility and strength.
   - **Occupational Therapy**: To help with daily living skills and regain independence.
   - **Speech and Language Therapy**: To address communication difficulties and swallowing issues.
   - **Neuropsychological Therapy**: To help with cognitive rehabilitation, including memory, attention, and problem-solving skills.

5. **Psychological Support**: 
   - Counseling or therapy may be beneficial for coping with emotional and psychological challenges following a brain injury, such as depression, anxiety, or changes in personality.

6. **Lifestyle Modifications**: 
   - Patients may need to make adjustments to their daily routines, including rest, nutrition, and avoiding activities that could lead to further injury.

7. **Supportive Care**: 
   - Family support and education about the injury and its effects can be crucial for recovery. Support groups may also be helpful.

8. **Long-term Management**: 
   - Ongoing follow-up with healthcare providers to monitor recovery and manage any long-term effects or complications.

9. **Assistive Devices**: 
   - Depending on the nature of the impairment, assistive devices or technology may be recommended to aid in communication, mobility, or daily activities.

It's important for treatment plans to be individualized, taking into account the specific needs and circumstances of the patient. A multidisciplinary team approach, involving neurologists, rehabilitation specialists, psychologists, and other healthcare professionals, is often the most effective way to address the complex needs of individuals with brain injuries.

**Observations:**
- The model correctly recognises the question as **traumatic brain injury (TBI)** and gives a broad treatment overview: emergency stabilisation and imaging (CT/MRI), surgery (hematoma evacuation, relieving pressure), medication, multidisciplinary rehabilitation (physical, occupational, speech, neuropsychological), psychological support and assistive devices.
- It contains a **clinically incorrect recommendation**: *"Corticosteroids may be used to reduce swelling in the brain."* Corticosteroids are **not recommended in TBI**, since large trials (e.g., CRASH) showed higher mortality. This shows how a base LLM can confidently state outdated or unsafe advice, a serious risk in a medical setting.
- It is **generic**: no severity classification (Glasgow Coma Scale), and no specific acute management steps such as preventing hypoxia and hypotension, head elevation, **intracranial pressure (ICP) monitoring**, or osmotic therapy (mannitol / hypertonic saline).
- The rehabilitation section is the strongest part, correctly describing a multidisciplinary team approach.
- The answer completed within the 800-token limit and cites **no source**, so the corticosteroid error can't be traced or checked, which reinforces the need for RAG grounded in the Merck Manual.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [30]:
# Write code to define the query above as user_input_5 and pass it to the response() function

# ==============================================================
# CELL 9: QUERY 5 - LEG FRACTURE ON A HIKING TRIP (base LLM, no instructions)
# ==============================================================

# --- Define the question exactly as given in the problem statement ---
user_input_5 = ("What are the necessary precautions and treatment steps for a person who has fractured "
                "their leg during a hiking trip, and what should be considered for their care and recovery?")

# --- Ask the base LLM (same settings as Queries 1-4 for a fair comparison) ---
answer_5 = response(user_input_5, max_tokens=800)

# --- Show the formatted answer ---
display(Markdown(answer_5))

If someone has fractured their leg during a hiking trip, it's crucial to take immediate and appropriate steps to ensure their safety and facilitate recovery. Here are the necessary precautions, treatment steps, and considerations for care and recovery:

### Immediate Precautions and Treatment Steps

1. **Ensure Safety**:
   - Move the injured person to a safe location away from any potential hazards (e.g., cliffs, wildlife).
   - If possible, have someone stay with the injured person while help is sought.

2. **Assess the Injury**:
   - Check for signs of a fracture, such as severe pain, swelling, deformity, or inability to bear weight.
   - Look for signs of circulation issues (e.g., pale or blue skin, numbness).

3. **Immobilization**:
   - Keep the leg as still as possible to prevent further injury.
   - If available, use a splint to immobilize the leg. You can create a makeshift splint using sticks, rolled-up newspapers, or any rigid material.
   - Secure the splint with cloth or bandages, but avoid wrapping it too tightly to prevent circulation issues.

4. **Control Bleeding**:
   - If there is an open wound, apply gentle pressure with a clean cloth or bandage to control bleeding.
   - Elevate the leg if possible to reduce swelling.

5. **Pain Management**:
   - If the person is conscious and not allergic, consider giving them over-the-counter pain relief (e.g., ibuprofen or acetaminophen) as per the recommended dosage.

6. **Seek Medical Help**:
   - Call for emergency medical services or arrange for transportation to the nearest medical facility.
   - If in a remote area, consider using a satellite phone or emergency beacon to call for help.

### Considerations for Care and Recovery

1. **Medical Evaluation**:
   - Once at a medical facility, the person will need an X-ray to confirm the fracture and assess its severity.
   - Treatment may involve casting, splinting, or in some cases, surgery.

2. **Follow Medical Advice**:
   - Adhere to the doctor’s recommendations regarding weight-bearing, physical therapy, and follow-up appointments.
   - Understand the importance of rest and not rushing the healing process.

3. **Rehabilitation**:
   - Engage in physical therapy as recommended to regain strength and mobility.
   - Follow a gradual return to normal activities, avoiding high-impact activities until cleared by a healthcare professional.

4. **Nutrition and Hydration**:
   - Maintain a balanced diet rich in calcium and vitamin D to support bone healing.
   - Stay hydrated, especially if recovery involves physical therapy.

5. **Monitor for Complications**:
   - Be aware of signs of complications such as increased pain, swelling, fever, or changes in skin color around the injury site.
   - Seek medical attention if any concerning symptoms arise.

6. **Emotional Support**:
   - A fracture can be emotionally challenging, especially if it limits mobility. Provide emotional support and encourage the person to engage in activities that can be done while recovering.

7. **Gradual Return to Activities**:
   - Once cleared by a healthcare provider, gradually reintroduce activities, starting with low-impact exercises and slowly increasing intensity.

By following these steps and considerations, the person can ensure a safer recovery process after a leg fracture sustained during a hiking trip.

**Observations:**
- The answer is practical and well organised into **immediate precautions** (safety, assessment, immobilisation with an improvised splint, bleeding control, pain relief, calling for help via satellite phone/beacon) and **care and recovery** (X-ray, casting or surgery, physical therapy, calcium/vitamin D, monitoring for complications, gradual return to activity).
- The **order of first steps is questionable**: it advises moving the person to a safe location *before* assessing and splinting the leg. Standard first aid is to **immobilise first and move only if there is immediate danger**, since moving an unsplinted fracture can worsen the injury.
- Important clinical red flags are missing: **open fracture** care (don't push bone back in, cover the wound, infection risk), **compartment syndrome** (severe, worsening pain and tight swelling), **shock** from blood loss in femur fractures, and **blood-clot (DVT) risk** during

## Question Answering using LLM with Prompt Engineering

In [31]:
# Write code to define a system_prompt string describing the assistant's role and response guidelines
# (e.g. specializing in medical knowledge, prioritizing factual correctness, acknowledging limitations, etc.)

# ==============================================================
# CELL 10: SYSTEM PROMPT - THE ASSISTANT'S ROLE AND RULES
# ==============================================================

# A system prompt is the "job description" we give the model before any question.
# It shapes tone, structure and caution, without changing the model itself.
system_prompt = """You are an experienced clinical assistant supporting doctors and nurses in a hospital.
Your answers are used by healthcare professionals, so accuracy and safety come first.

Follow these rules for every answer:
1. Be factually correct and follow current, evidence-based clinical guidelines.
   Do NOT recommend treatments that are outdated or shown to be harmful.
2. Structure the answer with clear headings, in the order a clinician would act:
   assessment/diagnosis -> immediate management -> definitive treatment -> monitoring/follow-up.
3. Where relevant, include specific, actionable details (doses, targets, time frames),
   and mention key red flags or complications to watch for.
4. If the question has several parts, answer every part explicitly.
5. Be concise: focus on what a clinician needs, avoid generic filler.
6. If evidence is uncertain or practice varies, say so clearly rather than guessing.
7. End with a one-line reminder that decisions must be based on clinical judgement
   and local protocols."""

print("System prompt defined.")
print(f"Length: {len(system_prompt.split())} words")

System prompt defined.
Length: 152 words


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [32]:
# Write code to combine system_prompt with the query above and pass it to the response() function

# ==============================================================
# CELL 11: QUERY 1 - SEPSIS (prompt engineering, combination 1)
# ==============================================================

# --- Combine the system prompt with the question into one prompt ---
# The rules come first, then the question, clearly labelled so the model
# knows which part is instructions and which part is the actual question.
prompt_1 = system_prompt + "\n\n###Question:\n" + user_input_1

# --- Combination 1: same settings as the base run (temperature=0, top_p=0.95, max_tokens=800) ---
# Keeping the settings identical isolates the effect of the prompt alone.
pe_answer_1 = response(prompt_1, max_tokens=800, temperature=0, top_p=0.95)

# --- Show the formatted answer ---
display(Markdown(pe_answer_1))

### Assessment/Diagnosis
1. **Identify Sepsis**: Use the Sepsis-3 criteria:
   - Confirm infection (e.g., pneumonia, urinary tract infection, etc.).
   - Assess for organ dysfunction using the SOFA score (≥2 points indicates sepsis).
   - Consider the presence of systemic inflammatory response syndrome (SIRS) criteria, though less emphasized in recent guidelines.

2. **Red Flags**: Look for signs of severe sepsis or septic shock:
   - Hypotension (systolic BP <90 mmHg or MAP <65 mmHg).
   - Altered mental status.
   - Respiratory distress (e.g., tachypnea, hypoxemia).
   - Oliguria or anuria.

### Immediate Management
1. **Resuscitation**:
   - Administer IV fluids (30 mL/kg of crystalloids within the first 3 hours).
   - Monitor hemodynamics closely (e.g., using a central venous catheter if necessary).

2. **Antibiotic Therapy**:
   - Initiate broad-spectrum IV antibiotics within 1 hour of sepsis recognition.
   - Choose based on local antibiogram and suspected source (e.g., piperacillin-tazobactam, ceftriaxone, or meropenem).

3. **Vasopressors**:
   - If hypotension persists after fluid resuscitation, initiate norepinephrine to maintain MAP ≥65 mmHg.

### Definitive Treatment
1. **Source Control**:
   - Identify and manage the source of infection (e.g., drainage of abscess, removal of infected devices).
   - Surgical intervention may be necessary depending on the source.

2. **Antibiotic Adjustment**:
   - Reassess antibiotic therapy based on culture results and clinical response, typically within 48-72 hours.

3. **Supportive Care**:
   - Provide organ support as needed (e.g., mechanical ventilation for respiratory failure, renal replacement therapy for acute kidney injury).

### Monitoring/Follow-Up
1. **Continuous Monitoring**:
   - Monitor vital signs, urine output, and laboratory parameters (e.g., lactate levels, complete blood count, renal function).
   - Reassess fluid status and hemodynamics frequently.

2. **Re-evaluation**:
   - Reassess the patient’s clinical status and response to treatment within 6 hours of initial management.
   - Adjust treatment based on clinical improvement or deterioration.

3. **Long-term Follow-up**:
   - Consider follow-up for potential post-sepsis syndrome, including physical and psychological rehabilitation.

### Key Complications to Watch For
- Multi-organ failure.
- Persistent hypotension despite adequate fluid resuscitation.
- Secondary infections due to prolonged ICU stay or antibiotic use.

Decisions must be based on clinical judgment and local protocols.

**Observations:**
- With the **same parameters** as the base run (temperature=0, top_p=0.95, max_tokens=800), the system prompt alone clearly improved the answer, so the gain comes from the prompt, not the settings.
- The response now follows the **requested clinical order** (Assessment/Diagnosis → Immediate Management → Definitive Treatment → Monitoring/Follow-Up), adds a **Key Complications** section, and ends with the requested clinical-judgement reminder.
- It is more **specific and current** than the base answer: **Sepsis-3 criteria (SOFA ≥ 2)**, explicit red-flag thresholds (SBP < 90 / MAP < 65 mmHg), **named empiric antibiotics** (piperacillin-tazobactam, ceftriaxone, meropenem) chosen by local antibiogram, and **antibiotic de-escalation at 48–72 hours**.
- Gaps remain: **corticosteroids** for refractory septic shock and **glucose control** are still missing, and it uses the outdated term "severe sepsis" (dropped in Sepsis-3).
- Prompt engineering improves structure and precision, but the model still answers **from memory with no source**, so it can't guarantee completeness against the Merck Manual. That's what RAG addresses.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [33]:
# Write code to combine system_prompt with the query above and pass it to the response() function

# ==============================================================
# CELL 12: QUERY 2 - APPENDICITIS (prompt engineering, combination 2)
# ==============================================================

# --- Combine the system prompt with the question ---
prompt_2 = system_prompt + "\n\n###Question:\n" + user_input_2

# --- Combination 2: slight variation (temperature=0.3, top_p=0.9, max_tokens=800) ---
# A small amount of randomness can make wording more natural without hurting accuracy.
pe_answer_2 = response(prompt_2, max_tokens=800, temperature=0.3, top_p=0.9)

# --- Show the formatted answer ---
display(Markdown(pe_answer_2))

### Assessment/Diagnosis
Common symptoms of appendicitis include:
- **Abdominal Pain**: Typically starts around the umbilicus and then shifts to the right lower quadrant (McBurney's point).
- **Nausea and Vomiting**: Often follows the onset of pain.
- **Loss of Appetite**: Patients may report a decreased desire to eat.
- **Fever**: Mild fever may be present.
- **Rebound Tenderness**: Pain upon release of pressure in the right lower quadrant.

**Key Red Flags**: 
- Severe pain that worsens over time.
- Signs of perforation (sudden relief of pain followed by worsening symptoms, high fever, tachycardia).

### Immediate Management
- **Initial Assessment**: Obtain a thorough history and perform a physical examination.
- **Laboratory Tests**: CBC may show leukocytosis; consider CRP and urinalysis to rule out urinary tract issues.
- **Imaging**: Ultrasound or CT scan of the abdomen can confirm diagnosis, especially in atypical presentations.

### Definitive Treatment
- **Surgical Procedure**: The standard treatment for appendicitis is an **appendectomy**, which can be performed via:
  - **Open Appendectomy**: Traditional method, larger incision.
  - **Laparoscopic Appendectomy**: Minimally invasive, preferred for quicker recovery and less postoperative pain.
  
**Timing**: Surgery is typically performed urgently, ideally within 24 hours of diagnosis to prevent complications like perforation.

### Monitoring/Follow-Up
- **Postoperative Care**: Monitor for signs of infection, bleeding, or complications such as abscess formation.
- **Follow-Up**: Schedule a follow-up appointment within 1-2 weeks post-surgery to assess recovery and wound healing.

### Conclusion
Appendicitis cannot be cured with medication alone; surgical intervention is necessary. Decisions must be based on clinical judgment and local protocols.

**Observations:**
- **Combination 2** (temperature=0.3, top_p=0.9, max_tokens=800): the slight randomness did not reduce accuracy; the answer stayed focused and followed the requested clinical structure.
- The prompt filled several gaps from the base answer: clinical signs (**McBurney's point, rebound tenderness**), **red flags for perforation**, a **diagnostic workup** (CBC for leukocytosis, CRP, urinalysis, **ultrasound/CT**), surgical **timing (ideally within 24 hours)**, and **follow-up at 1–2 weeks**.
- All three parts of the question were answered explicitly, showing rule 4 ("answer every part") worked.
- It **still states appendicitis "cannot be cured with medication alone"** and omits antibiotic-only treatment for selected uncomplicated cases, even though rule 6 asked it to flag varying practice. **Prompting can shape how the model answers, but it can't add knowledge the model doesn't reliably have.**
- Pre-operative antibiotics are still not mentioned, and the answer cites **no source**, so RAG is needed to ground it in the Merck Manual.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [34]:
# Write code to combine system_prompt with the query above and pass it to the response() function

# ==============================================================
# CELL 13: QUERY 3 - HAIR LOSS (prompt engineering, combination 3)
# ==============================================================

# --- Combine the system prompt with the question ---
prompt_3 = system_prompt + "\n\n###Question:\n" + user_input_3

# --- Combination 3: high creativity (temperature=0.7, top_p=0.9, max_tokens=800) ---
# Testing whether more randomness makes the medical answer less precise or less focused.
pe_answer_3 = response(prompt_3, max_tokens=800, temperature=0.7, top_p=0.9)

# --- Show the formatted answer ---
display(Markdown(pe_answer_3))

### Assessment/Diagnosis

**Possible Causes:**
1. **Alopecia Areata**: An autoimmune condition leading to sudden hair loss in patches.
2. **Tinea Capitis**: Fungal infection of the scalp, often presenting with localized hair loss and scaling.
3. **Telogen Effluvium**: Stress-induced hair loss that can present as patchy areas.
4. **Traction Alopecia**: Hair loss due to pulling or tension on the hair.
5. **Other Dermatological Conditions**: Psoriasis, eczema, or contact dermatitis may cause localized hair loss.

**Diagnostic Steps:**
- **History and Physical Examination**: Assess for patterns of hair loss, family history, and any associated symptoms (itching, pain).
- **Scalp Examination**: Look for signs of inflammation, scaling, or scarring.
- **Laboratory Tests**: Consider a scalp biopsy or fungal culture if tinea capitis or other conditions are suspected.

### Immediate Management

1. **Alopecia Areata**:
   - **Topical Corticosteroids**: Clobetasol propionate 0.05% cream applied once daily to affected areas.
   - **Intralesional Corticosteroids**: Triamcinolone acetonide (5-10 mg/mL) injected every 4-6 weeks for more extensive patches.

2. **Tinea Capitis**:
   - **Oral Antifungals**: Griseofulvin 20 mg/kg/day for 6-12 weeks, or terbinafine 250 mg daily for 4-6 weeks, depending on the severity and age of the patient.

3. **Telogen Effluvium**:
   - **Supportive Care**: Reassurance and addressing underlying stressors. Nutritional support may be beneficial.

4. **Traction Alopecia**:
   - **Behavior Modification**: Advise on reducing hair tension and using protective hairstyles.

### Definitive Treatment

- **Alopecia Areata**: In cases resistant to topical or intralesional corticosteroids, consider:
  - **Systemic Corticosteroids**: Prednisone at a tapering dose (starting at 0.5-1 mg/kg/day).
  - **Immunotherapy**: Topical diphencyprone (DPCP) for extensive cases.
  
- **Tinea Capitis**: Completion of oral antifungal course; monitor for resolution of symptoms.

- **Telogen Effluvium**: Generally self-resolving; treat underlying causes if identifiable.

### Monitoring/Follow-up

- **Alopecia Areata**: Follow up every 4-6 weeks to assess treatment response and adjust therapy as needed.
- **Tinea Capitis**: Monitor for signs of resolution and potential recurrence.
- **Telogen Effluvium**: Follow up in 3-6 months to evaluate hair regrowth and address any ongoing stressors.

### Red Flags and Complications
- Persistent or worsening hair loss, signs of secondary infection (in tinea capitis), or systemic symptoms (fever, malaise) warrant immediate reevaluation.
  
Decisions must be based on clinical judgement and local protocols.

**Observations:**
- **Combination 3** (temperature=0.7, top_p=0.9, max_tokens=800): despite higher randomness, the answer stayed well structured and followed the requested clinical order.
- It fixed the base answer's main flaw by treating each condition **separately**: alopecia areata, tinea capitis, telogen effluvium and traction alopecia each get their own management, instead of one mixed list of "causes".
- It added a **diagnostic workup** (scalp exam, fungal culture, biopsy) and **specific doses**: clobetasol 0.05%, intralesional triamcinolone 5–10 mg/mL every 4–6 weeks, griseofulvin 20 mg/kg/day, terbinafine 250 mg/day, and prednisone 0.5–1 mg/kg/day for resistant alopecia areata.
- It contains a **factual slip**: telogen effluvium is described as "patchy", but it typically causes **diffuse** thinning. Minoxidil and trichotillomania, both in the base answer, were dropped. This is the kind of drift higher temperature can introduce.
- The doses look plausible but are **unverifiable without a source**, which is risky in a clinical setting; precise figures need grounding in a trusted reference like the Merck Manual.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [35]:
# Write code to combine system_prompt with the query above and pass it to the response() function

# ==============================================================
# CELL 14: QUERY 4 - BRAIN INJURY (prompt engineering, combination 4)
# ==============================================================

# --- Combine the system prompt with the question ---
prompt_4 = system_prompt + "\n\n###Question:\n" + user_input_4

# --- Combination 4: focused but detailed (temperature=0.2, top_p=0.8, max_tokens=1000) ---
# Low randomness plus a narrower word pool (top_p=0.8) for precision,
# and a larger token budget so the acute + rehab phases both fit.
pe_answer_4 = response(prompt_4, max_tokens=1000, temperature=0.2, top_p=0.8)

# --- Show the formatted answer ---
display(Markdown(pe_answer_4))

### Assessment/Diagnosis
1. **Initial Evaluation**: Conduct a thorough neurological examination to assess consciousness, cognitive function, motor skills, and sensory responses. Use the Glasgow Coma Scale (GCS) to quantify the level of consciousness.
2. **Imaging**: Obtain a CT scan of the head to identify any intracranial hemorrhage, contusions, or other structural injuries. An MRI may be considered for further evaluation if indicated.
3. **Classification**: Determine the severity of the injury (mild, moderate, severe) based on GCS scores and imaging findings.

### Immediate Management
1. **Stabilization**: Ensure airway, breathing, and circulation (ABCs) are stable. Administer oxygen if hypoxia is present.
2. **Intravenous Access**: Establish IV access for fluid resuscitation and medication administration.
3. **Monitoring**: Continuous monitoring of vital signs and neurological status is critical. Use a neuromonitoring protocol if necessary.
4. **Seizure Prophylaxis**: Consider administering anticonvulsants (e.g., levetiracetam) if there is a high risk of seizures, particularly in cases of moderate to severe TBI.

### Definitive Treatment
1. **Surgical Intervention**: If there is evidence of significant mass effect (e.g., hematoma) or elevated intracranial pressure (ICP), surgical intervention (e.g., craniotomy or craniectomy) may be necessary.
2. **Medical Management**: 
   - **ICP Management**: Use osmotic agents like mannitol or hypertonic saline to reduce ICP.
   - **Neuroprotective Strategies**: Maintain normothermia, manage blood glucose levels (target 140-180 mg/dL), and ensure adequate cerebral perfusion pressure (CPP).
3. **Rehabilitation**: Initiate early rehabilitation services (physical, occupational, speech therapy) as soon as the patient is stable to optimize recovery.

### Monitoring/Follow-Up
1. **Neurological Assessment**: Regularly reassess neurological status and GCS score. Monitor for signs of deterioration or complications (e.g., worsening headache, vomiting, seizures).
2. **Follow-Up Imaging**: Repeat imaging may be necessary if clinical status changes or if there are concerns about complications.
3. **Long-Term Rehabilitation**: Coordinate with rehabilitation specialists for ongoing therapy and support, especially for patients with persistent cognitive or physical impairments.

### Key Red Flags/Complications
- Worsening headache or confusion
- New focal neurological deficits
- Signs of infection (fever, increased ICP)
- Seizures

Decisions must be based on clinical judgment and local protocols.

**Observations:**
- **Combination 4** (temperature=0.2, top_p=0.8, max_tokens=1000): low randomness with a narrower word pool produced the most precise, protocol-like answer so far, and the larger token budget let it cover acute care through rehabilitation without being cut off.
- The **incorrect corticosteroid recommendation from the base answer is gone**, showing rule 1 ("do not recommend treatments shown to be harmful") directly corrected a clinically unsafe output.
- It adds the key elements the base answer lacked: **Glasgow Coma Scale** for severity, **CT head** first, **ABC stabilisation**, **seizure prophylaxis (levetiracetam)**, **craniotomy/craniectomy** for mass effect, **osmotic therapy (mannitol / hypertonic saline)** for raised ICP, and neuroprotective measures (normothermia, glucose 140–180 mg/dL, cerebral perfusion pressure).
- Some specifics are still missing: no numeric **blood-pressure or CPP targets**, no **head-of-bed elevation**, and the psychological support from the base answer was dropped. It also oddly groups "increased ICP" under signs of infection.
- The answer is safer and more clinically useful, but it still cites **no source** for its targets and drug choices.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [36]:
# Write code to combine system_prompt with the query above and pass it to the response() function

# ==============================================================
# CELL 15: QUERY 5 - LEG FRACTURE (prompt engineering, combination 5)
# ==============================================================

# --- Combine the system prompt with the question ---
prompt_5 = system_prompt + "\n\n###Question:\n" + user_input_5

# --- Combination 5: medium creativity, tighter length (temperature=0.5, top_p=1.0, max_tokens=600) ---
# top_p=1.0 lets the model consider its full vocabulary, while the smaller
# token budget tests whether it can stay complete when forced to be concise.
pe_answer_5 = response(prompt_5, max_tokens=600, temperature=0.5, top_p=1.0)

# --- Show the formatted answer ---
display(Markdown(pe_answer_5))

### Assessment/Diagnosis
1. **Initial Assessment**: Evaluate the patient's vital signs (heart rate, blood pressure, respiratory rate) and level of consciousness. Check for signs of shock (e.g., pallor, tachycardia).
2. **Physical Examination**: Inspect the injured leg for deformity, swelling, bruising, and open wounds. Assess for neurovascular status (capillary refill, pulse, sensation, and movement distal to the injury).
3. **Imaging**: If possible, obtain X-rays to confirm the fracture type (e.g., closed vs. open, type of fracture). 

### Immediate Management
1. **Stabilization**: 
   - If the fracture is open, cover the wound with a sterile dressing to prevent infection.
   - Immobilize the leg using a splint or by applying traction if necessary.
2. **Pain Management**: Administer analgesics such as acetaminophen or NSAIDs (e.g., ibuprofen) as per dosing guidelines.
3. **Transport**: Arrange for safe transport to a medical facility, ensuring the leg remains immobilized.

### Definitive Treatment
1. **Orthopedic Consultation**: Refer to an orthopedic surgeon for evaluation and treatment.
2. **Surgical Intervention**: Depending on the fracture type and displacement:
   - Closed fractures may require casting.
   - Open or unstable fractures may require surgical fixation (e.g., plates, screws).
3. **Antibiotics**: If the fracture is open, initiate prophylactic antibiotics (e.g., cefazolin) as per local guidelines.

### Monitoring/Follow-Up
1. **Post-Operative Care**: Monitor for signs of infection (increased pain, swelling, fever) and complications such as compartment syndrome.
2. **Follow-Up Appointments**: Schedule follow-up visits for X-rays to assess healing and adjust treatment as needed.
3. **Rehabilitation**: Initiate physical therapy as appropriate to restore function and strength once healing allows.

### Considerations for Care and Recovery
- **Weight Bearing**: Follow orthopedic advice regarding weight-bearing status (e.g., non-weight bearing, partial weight bearing).
- **Nutrition**: Ensure adequate nutrition to support bone healing (calcium, vitamin D, protein).
- **Patient Education**: Educate the patient on signs of complications and the importance of adherence to rehabilitation protocols.

### Reminder
Decisions must be based on clinical judgement and local protocols.

**Observations:**
- **Combination 5** (temperature=0.5, top_p=1.0, max_tokens=600): the answer **completed within 600 tokens**, so the tighter budget forced a concise answer without truncation.
- The **first-aid order is now correct**: assess (vitals, signs of shock, neurovascular status), then **immobilise before transport**, with the leg kept immobilised during transfer. This fixes the base answer's "move first" advice.
- It adds the clinical red flags the base answer missed: **open-fracture care** (sterile dressing, prophylactic **cefazolin**), **shock**, and **compartment syndrome** monitoring.
- It **lost the hiking context**: it reads like a hospital protocol ("obtain X-rays" under initial assessment), and the practical wilderness advice from the base answer (improvised splints, satellite phone/beacon) was dropped. **DVT risk** during immobilisation is still missing.
- Across the 5 combinations, the **system prompt drove most of the improvement** (structure, safety, specificity), while parameters mainly affected length and consistency; lower temperatures (0–0.3) gave the most reliable answers for clinical use.

## Data Preparation for RAG

### Loading the Data

In [5]:
# ==============================================================
# CELL R1: LOAD THE MEDICAL MANUAL (PDF)
# Data Preparation for RAG
# ==============================================================

# --- Define the file path ---
# The PDF sits in the same folder as this notebook, so the plain file name works.
manual_pdf_path = "medical_diagnosis_manual.pdf"

# --- Safety check: make sure the file is actually there ---
# Without this, a missing file gives a confusing error deep inside the library.
# This way we get a clear, plain-English message instead.
if not os.path.exists(manual_pdf_path):
    raise FileNotFoundError(f"Can't find '{manual_pdf_path}'. Put it in the same folder as this notebook.")

# --- Create the PDF loader ---
# PyMuPDFLoader is fast and keeps the page number of each page as metadata,
# which lets us trace every answer back to its source page later.
pdf_loader = PyMuPDFLoader(manual_pdf_path)

# --- Load all pages into memory ---
# .load() returns a list: one "Document" per PDF page.
# Each Document has .page_content (the text) and .metadata (page number, source, etc.).
# With 4,000+ pages this can take 1-2 minutes, so wait for the [*] to finish.
manual = pdf_loader.load()

# --- Confirm it worked ---
print("PDF loaded successfully!")
print("Pages loaded:", len(manual))

PDF loaded successfully!
Pages loaded: 4114


### Data Overview

#### Checking the first 5 pages

In [ ]:
# ==============================================================
# CELL R2: CHECK THE FIRST 5 PAGES
# Data Preparation for RAG - Data Overview
# ==============================================================

# --- Loop through the first 5 pages ---
# range(5) gives 0, 1, 2, 3, 4 -> Python counts from 0.
# We add 1 when printing so the page numbers match the PDF (page 1, 2, 3...).
for i in range(5):

    # Each item in 'manual' is a Document holding one PDF page.
    page = manual[i]

    # Print a clear header so each page is easy to spot in the output.
    print("=" * 80)
    print(f"PAGE {i + 1}")
    print("=" * 80)

    # .page_content holds the raw text extracted from that page.
    print(page.page_content)

    # Blank line between pages for readability.
    print()

   *R2 output cleared before publishing: the source PDF carries a personal-use watermark.*

#### Checking the number of pages

In [7]:
# ==============================================================
# CELL R3: CHECK THE TOTAL NUMBER OF PAGES
# Data Preparation for RAG - Data Overview
# ==============================================================

# len() counts how many Documents are in the list.
# Since the loader creates one Document per PDF page, this equals the page count.
total_pages = len(manual)

# Print the result, with a thousands separator (:,) for readability -> 4,114
print(f"Total number of pages in the manual: {total_pages:,}")

Total number of pages in the manual: 4,114


**Observations:**
- The Merck Manual loaded successfully with **4,114 pages**, one LangChain Document per page, each carrying its page number as metadata for source tracing.
- Pages 1–2 contain no medical content (only a watermark); page 3 onward is the table of contents, which is mostly dotted leader lines (`.....`) and page numbers.
- A personal-use watermark (email, code and legal notice) appears on **every page**. Left in, it would be embedded into every chunk as repeated noise, so it will be removed, along with the dotted lines, before chunking.

### Data Cleaning

In [14]:
# ==============================================================
# CELL R4: CLEAN THE PAGE TEXT
# Remove the watermark and dotted leader lines before chunking
# ==============================================================

# --- Define what counts as watermark ---
# These two lines appear EXACTLY like this on every page.
watermark_lines = [
    "krajbhandari3@gmail.com",
    "L5BQ932MN6",
]

# These two sentences are also on every page; we match on how they START,
# which is safer than matching the full sentence character-for-character.
watermark_starts = [
    "This file is meant for personal use",
    "Sharing or publishing the contents",
]

# --- Set up containers and counters ---
cleaned_manual = []        # will hold only pages that still have content after cleaning
chars_before = 0           # total characters before cleaning (to measure impact)
chars_after = 0            # total characters after cleaning

# --- Clean page by page ---
for page in manual:

    original_text = page.page_content
    chars_before = chars_before + len(original_text)

    # Split the page into lines so we can check each one individually.
    kept_lines = []
    for line in original_text.split("\n"):

        stripped = line.strip()              # ignore leading/trailing spaces when comparing

        # Skip lines that exactly match a watermark line.
        if stripped in watermark_lines:
            continue

        # Skip lines that start with a watermark sentence.
        is_watermark = False
        for start in watermark_starts:
            if stripped.startswith(start):
                is_watermark = True
        if is_watermark:
            continue

        # If we got here, the line is real content -> keep it.
        kept_lines.append(line)

    # Glue the kept lines back into one block of text.
    cleaned_text = "\n".join(kept_lines)

    # Replace any run of 4+ dots (table-of-contents leaders) with a single space.
    # \.{4,} means "a literal dot, repeated 4 or more times".
    # We stop at 4 so normal text like "..." (ellipsis) is left alone.
    cleaned_text = re.sub(r"\.{4,}", " ", cleaned_text)

    # Remove blank space at the start and end of the page.
    cleaned_text = cleaned_text.strip()

    # If nothing is left (e.g. the cover and blank page), drop the page entirely.
    # An empty page would only create empty, useless chunks.
    if cleaned_text == "":
        continue

    # Save the cleaned text back onto the Document. Metadata (page number) is untouched.
    page.page_content = cleaned_text
    cleaned_manual.append(page)
    chars_after = chars_after + len(cleaned_text)

# --- Replace the original list with the cleaned one ---
# The chunking step later expects a variable called 'manual', so we reuse that name.
pages_before = len(manual)
manual = cleaned_manual

# --- Report the impact ---
percent_removed = (chars_before - chars_after) / chars_before * 100
print(f"Pages before cleaning : {pages_before:,}")
print(f"Pages after cleaning  : {len(manual):,}  (empty pages removed: {pages_before - len(manual)})")
print(f"Characters removed    : {chars_before - chars_after:,}  ({percent_removed:.1f}% of all text)")

# --- Spot check: show the start of the first remaining page ---
print("\nFirst 300 characters of the first page after cleaning:")
print("-" * 60)
print(manual[0].page_content[:300])

Pages before cleaning : 4,114
Pages after cleaning  : 4,112  (empty pages removed: 2)
Characters removed    : 805,526  (5.9% of all text)

First 300 characters of the first page after cleaning:
------------------------------------------------------------
Table of Contents
1
Front     
1
Cover     
2
Front Matter     
53
1 - Nutritional Disorders     
53
Chapter 1. Nutrition: General Considerations     
59
Chapter 2. Undernutrition     
69
Chapter 3. Nutritional Support     
76
Chapter 4. Vitamin Deficiency, Dependency & Toxicity     
99
Chapter 5. M


**Observations:**
- Cleaning removed **805,526 characters (5.9% of all text)**: the watermark on every page plus the table-of-contents dotted lines.
- **2 pages** (cover and blank page) contained only the watermark and were dropped, leaving **4,112 pages** of real content.
- Page-number metadata was preserved, so every answer can still be traced back to its source page in the manual.
- Without this step, the same ~40 words of watermark would appear in every chunk, adding noise to similarity search and wasting tokens on every LLM call.

### Data Chunking

In [15]:
# Write code to:
#   - create a RecursiveCharacterTextSplitter (using the tiktoken encoder) with an appropriate chunk_size and chunk_overlap
#   - split the manual into document_chunks using this splitter
#   - print the total number of chunks created
#   - view the content of a few chunks (e.g. document_chunks[0], [2], [3]) to check for overlaps

# ==============================================================
# CELL R5: SPLIT THE MANUAL INTO CHUNKS
# Data Preparation for RAG - Data Chunking
# ==============================================================

# --- Create the text splitter ---
# from_tiktoken_encoder() measures chunk size in TOKENS (how models read text),
# not characters, so our sizes line up with the embedding model's limits.
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",   # the tokenizer used by modern OpenAI models
    chunk_size=400,                # max tokens per chunk (~300 words, a few paragraphs)
    chunk_overlap=50               # tokens repeated between neighbouring chunks
)

# --- Split all cleaned pages into chunks ---
# split_documents() keeps each chunk's metadata (page number, source),
# so every chunk still knows which page of the manual it came from.
document_chunks = text_splitter.split_documents(manual)

# --- How many chunks did we get? ---
print(f"Total number of chunks created: {len(document_chunks):,}")

# --- View a few chunks to check the overlap ---
# Chunks 2 and 3 are neighbours, so the END of chunk 2 should
# reappear at the START of chunk 3. That repeated text is the overlap.
for i in [0, 2, 3]:
    print("\n" + "=" * 80)
    print(f"CHUNK {i}  |  from page {document_chunks[i].metadata['page'] + 1}")
    print("=" * 80)
    print(document_chunks[i].page_content)

Total number of chunks created: 10,430

CHUNK 0  |  from page 3
Table of Contents
1
Front     
1
Cover     
2
Front Matter     
53
1 - Nutritional Disorders     
53
Chapter 1. Nutrition: General Considerations     
59
Chapter 2. Undernutrition     
69
Chapter 3. Nutritional Support     
76
Chapter 4. Vitamin Deficiency, Dependency & Toxicity     
99
Chapter 5. Mineral Deficiency & Toxicity     
108
Chapter 6. Obesity & the Metabolic Syndrome     
120
2 - Gastrointestinal Disorders     
120
Chapter 7. Approach to the Patient With Upper GI Complaints     
132
Chapter 8. Approach to the Patient With Lower GI Complaints     
143
Chapter 9. Diagnostic & Therapeutic GI Procedures     
150
Chapter 10. GI Bleeding     
158
Chapter 11. Acute Abdomen & Surgical Gastroenterology     
172
Chapter 12. Esophageal & Swallowing Disorders     
183
Chapter 13. Gastritis & Peptic Ulcer Disease     
196
Chapter 14. Bezoars & Foreign Bodies     
199
Chapter 15. Pancreatitis     
206
Chapter 16. Gastroenter

**Observations:**
- The 4,112 cleaned pages were split into **10,430 chunks** (about 2.5 chunks per page) using a token-based splitter with `chunk_size=400` and `chunk_overlap=50`.
- Chunk size is kept at 400 tokens so each chunk stays under the embedding model's 512-token limit; anything longer would be silently truncated.
- The overlap works as intended: the text at the end of chunk 2 (*Chapter 70. Orbital Diseases … 7 - Dermatologic Disorders*) reappears in chunk 3, so ideas that span a chunk boundary aren't cut in half.
- Every chunk keeps its **page number** metadata (e.g. chunk 3 → page 4), so retrieved answers can be traced back to the source page.
- The first chunks come from the table of contents and carry little medical value; they're a small share of 10,430 chunks and unlikely to be retrieved for clinical questions.

### Embedding

In [70]:
# Write code to:
#   - create an embedding_model using HuggingFaceEmbeddings with an appropriate model name
#   - generate embeddings for two document chunks and check that their dimensions match

# ==============================================================
# CELL R6: CREATE THE EMBEDDING MODEL
# Data Preparation for RAG - Embedding
# ==============================================================

# --- Choose where the model runs ---
# Your laptop has an NVIDIA GPU, so use it ("cuda"); fall back to CPU if not.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding model will run on:", device)

# --- Hide the "Loading weights" progress bar (keeps the exported HTML clean) ---
# These switch off the progress bars that transformers / huggingface_hub
# print to the error stream, which shows up as a pink box in the notebook.
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
hf_logging.disable_progress_bar()
disable_progress_bars()

# --- Load the embedding model ---
# gte-large is a strong, free, open-source embedding model:
#   - turns any text into a list of 1,024 numbers (a "vector")
#   - reads up to 512 tokens per chunk, which is why chunk_size = 400
embedding_model = HuggingFaceEmbeddings(
    model_name="thenlper/gte-large",
    model_kwargs={"device": device},                 # run on GPU for speed
    encode_kwargs={"normalize_embeddings": True}     # scale vectors to length 1 for fair similarity comparison
)

# --- Embed two different chunks ---
# embed_query() converts one piece of text into its vector.
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

# --- Check that both vectors have the same size ---
# Every vector MUST have the same length, or the database can't compare them.
print("Dimension of embedding 1:", len(embedding_1))
print("Dimension of embedding 2:", len(embedding_2))
print("Dimensions match:", len(embedding_1) == len(embedding_2))

# --- Peek at the first few numbers of one vector ---
print("\nFirst 5 values of embedding 1:", embedding_1[:5])

Embedding model will run on: cuda
Dimension of embedding 1: 1024
Dimension of embedding 2: 1024
Dimensions match: True

First 5 values of embedding 1: [-0.005817413330078125, -0.037109375, 0.01812744140625, 0.00598907470703125, -0.04290771484375]


### Vector Database

In [18]:
# ==============================================================
# CELL R7: CREATE THE FOLDER FOR THE VECTOR DATABASE
# Data Preparation for RAG - Vector Database
# ==============================================================

# --- Name of the folder where Chroma will save the database ---
# Saving to disk ("persisting") means we can reload the database later
# without re-embedding all 10,430 chunks every time.
out_dir = "medical_db"

# --- Create the folder only if it doesn't already exist ---
# os.makedirs() would raise an error if the folder is already there,
# so we check first. This makes the cell safe to run more than once.
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

print("Vector database folder ready:", os.path.abspath(out_dir))

Vector database folder ready: D:\AI @ UT Austin\AIML\Python\05 Natural Language Processing\Project\medical_db


In [19]:
# Write code to:
#   - create a Chroma vectorstore from document_chunks and embedding_model, persisted to out_dir
#   - reload the vectorstore from out_dir
#   - run a similarity_search on the vectorstore with a sample query and an appropriate k value

# ==============================================================
# CELL R8: BUILD, RELOAD AND TEST THE CHROMA VECTOR DATABASE
# Data Preparation for RAG - Vector Database
# ==============================================================

# --- Build the database only if the folder is empty ---
# Embedding 10,430 chunks takes a few minutes. If the database was already
# built and saved on a previous run, we skip this step and just reload it.
# This also stops duplicate chunks being added if the cell is run twice.
if len(os.listdir(out_dir)) == 0:
    print("Building the vector database (this takes a few minutes on GPU)...")
    vectorstore = Chroma.from_documents(
        documents=document_chunks,      # the 10,430 chunks from R5
        embedding=embedding_model,      # gte-large model from R6
        persist_directory=out_dir       # save everything to the medical_db folder
    )
    print("Database built and saved.")
else:
    print("Database already exists in", out_dir, "-> skipping the build.")

# --- Reload the database from disk ---
# This proves the database was saved correctly and can be reused later.
vectorstore = Chroma(
    persist_directory=out_dir,
    embedding_function=embedding_model  # MUST be the same model used to build it
)
print("Chunks stored in the database:", vectorstore._collection.count())

# --- Test it with a sample medical question ---
# similarity_search() embeds the question and returns the k closest chunks.
sample_query = "What are the common symptoms of appendicitis?"
results = vectorstore.similarity_search(sample_query, k=3)

# --- Show where each result came from and a preview of its text ---
for i in range(len(results)):
    doc = results[i]
    print("\n" + "=" * 80)
    print(f"RESULT {i + 1}  |  page {doc.metadata['page'] + 1}")
    print("=" * 80)
    print(doc.page_content[:400])   # first 400 characters is enough to judge relevance

Building the vector database (this takes a few minutes on GPU)...
Database built and saved.
Chunks stored in the database: 10430

RESULT 1  |  page 174
Etiology
Appendicitis is thought to result from obstruction of the appendiceal lumen, typically by lymphoid
hyperplasia, but occasionally by a fecalith, foreign body, or even worms. The obstruction leads to
distention, bacterial overgrowth, ischemia, and inflammation. If untreated, necrosis, gangrene, and
perforation occur. If the perforation is contained by the omentum, an appendiceal abscess res

RESULT 2  |  page 174
occur. Pain may not be localized, particularly in infants and children. Tenderness may be diffuse or, in rare
instances, absent. Bowel movements are usually less frequent or absent; if diarrhea is a sign, a
retrocecal appendix should be suspected. RBCs or WBCs may be present in the urine. Atypical symptoms
are common among elderly patients and pregnant women; in particular, pain is less severe and

RESULT 3  |  page 168


**Observations:**

- All **10,430 chunks** were embedded with `gte-large` and stored in a persisted Chroma database (`medical_db`), which reloads from disk in seconds instead of re-embedding each run.
- A sample query, *"What are the common symptoms of appendicitis?"*, with `k=3` returned highly relevant chunks: two from **page 174** (appendicitis etiology and clinical symptoms) and one from **page 168** (the Acute Abdomen chapter introduction).
- Retrieval works on meaning, not just keywords: the top results describe pain, tenderness and atypical presentations even though the wording differs from the query.
- The third result is broader context rather than a direct answer, which suggests a slightly larger `k` could help capture treatment details for multi-part questions.
- Building the database took ~30+ minutes on a GTX 1660 Ti (a one-time cost); the persistent store avoids repeating it.


### Retriever

In [20]:
# ==============================================================
# CELL R9: CREATE THE RETRIEVER
# Data Preparation for RAG - Retriever
# ==============================================================

# --- Turn the vector database into a retriever ---
# search_type="similarity": return the chunks whose meaning is closest to the question.
# k=3: return the top 3 chunks. Enough context to answer, small enough to keep
#      the prompt short (fewer tokens = faster and cheaper LLM calls).
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# --- Test it with one of the project's own questions ---
# Using a different question from R8 shows the retriever generalises.
sample_query = "What is the protocol for managing sepsis in a critical care unit?"
relevant_chunks = retriever.invoke(sample_query)

print(f"Chunks retrieved: {len(relevant_chunks)}")

# --- Show where each chunk came from and a preview ---
for i in range(len(relevant_chunks)):
    doc = relevant_chunks[i]
    print("\n" + "=" * 80)
    print(f"CHUNK {i + 1}  |  page {doc.metadata['page'] + 1}")
    print("=" * 80)
    print(doc.page_content[:400])

Chunks retrieved: 3

CHUNK 1  |  page 2401
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special
populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high

CHUNK 2  |  page 2996
sepsis, initial therapy should include vancomycin (active against methicillin-resistant S. aureus) plus an
aminoglycoside. If P. aeruginosa is prevalent in the nursery, ceftazidime may be used instead of an
aminoglycoside. For neonates previously treated with a full 7- to 14-day aminoglycoside course who need
retreatment, a different aminoglycoside or a 3rd-generation cephalosporin should be consi

CHUNK 3  |  page 2456
metabolic acidosis becomes established, especially in conjunction with multiorgan failure, septic shock is
li

**Observations:**


- The retriever (`search_type="similarity"`, `k=3`) returned 3 chunks for *"What is the protocol for managing sepsis in a critical care unit?"*, with mixed relevance.
- **Chunk 3 (page 2456)** is highly relevant: it lists septic shock treatment (fluid resuscitation with 0.9% saline, O₂, broad-spectrum antibiotics, source control, glucose control, corticosteroids).
- **Chunk 1 (page 2401)** is a general introduction to critical care medicine: related to the ICU setting but not sepsis-specific.
- **Chunk 2 (page 2996)** covers sepsis therapy in **neonates**: semantically close ("sepsis", "initial therapy") but the wrong patient population for an adult ICU question.
- Only 1 of 3 chunks directly answers the question, so `k=3` may be too narrow for protocol-style questions. Larger `k` values will be tested in the fine-tuning section.


In [38]:
# Write code to call the OpenAI client directly (client.chat.completions.create) with the query:
# "What treatment options are available for managing hypertension?" -- without using any retrieved context

# ==============================================================
# CELL R10: HYPERTENSION QUESTION WITHOUT RETRIEVED CONTEXT
# The "before" picture: the LLM answers from memory only
# ==============================================================

# --- Call the OpenAI client directly (no retriever, no manual) ---
no_context_response = client.chat.completions.create(
    model=llm,                        # gpt-4o-mini
    messages=[{"role": "user",
               "content": "What treatment options are available for managing hypertension?"}],
    max_tokens=800,                   # enough room so the answer isn't cut off (512 was too small)
    temperature=0,                    # focused, repeatable answer
)

# --- Pull out and display just the answer text ---
no_context_answer = no_context_response.choices[0].message.content
display(Markdown(no_context_answer))

Managing hypertension (high blood pressure) typically involves a combination of lifestyle changes and medication. Here are the main treatment options:

### Lifestyle Changes
1. **Dietary Modifications**:
   - **DASH Diet**: Emphasizes fruits, vegetables, whole grains, and low-fat dairy while reducing saturated fat and cholesterol.
   - **Reduce Sodium Intake**: Aim for less than 2,300 mg per day, or ideally 1,500 mg for greater blood pressure reduction.
   - **Limit Alcohol**: Moderate consumption is advised (up to one drink per day for women and two for men).

2. **Physical Activity**:
   - Engage in regular aerobic exercise (e.g., walking, jogging, cycling) for at least 150 minutes per week.

3. **Weight Management**:
   - Achieving and maintaining a healthy weight can significantly lower blood pressure.

4. **Stress Management**:
   - Techniques such as meditation, deep breathing exercises, and yoga can help reduce stress levels.

5. **Quit Smoking**:
   - Smoking cessation can improve overall cardiovascular health and lower blood pressure.

### Medications
If lifestyle changes are insufficient, healthcare providers may prescribe medications. Common classes of antihypertensive drugs include:

1. **Diuretics**: Help the body eliminate excess sodium and water, reducing blood volume.
   - Examples: Hydrochlorothiazide, Chlorthalidone.

2. **ACE Inhibitors**: Help relax blood vessels by inhibiting the formation of a hormone that narrows blood vessels.
   - Examples: Lisinopril, Enalapril.

3. **Angiotensin II Receptor Blockers (ARBs)**: Block the action of the hormone that narrows blood vessels.
   - Examples: Losartan, Valsartan.

4. **Calcium Channel Blockers**: Prevent calcium from entering heart and blood vessel cells, leading to relaxed blood vessels.
   - Examples: Amlodipine, Diltiazem.

5. **Beta-Blockers**: Reduce heart rate and the heart's workload.
   - Examples: Metoprolol, Atenolol.

6. **Renin Inhibitors**: Directly inhibit renin, an enzyme involved in blood pressure regulation.
   - Example: Aliskiren.

7. **Alpha-Blockers**: Help reduce nerve signals to blood vessels, allowing them to remain open.
   - Examples: Doxazosin, Prazosin.

### Combination Therapy
In many cases, a combination of medications may be necessary to achieve optimal blood pressure control.

### Regular Monitoring
Regular blood pressure monitoring is essential to assess the effectiveness of treatment and make necessary adjustments.

### Consultation with Healthcare Provider
It's important for individuals with hypertension to work closely with their healthcare provider to develop a personalized treatment plan that considers their overall health, any other medical conditions, and potential side effects of medications.

Always consult a healthcare professional before making any changes to your treatment plan.

**Observations:**
- Called directly through the OpenAI client with `max_tokens=800`, the answer **completed in full**, unlike the earlier 512-token test that was cut off mid-word, and now adds combination therapy, regular monitoring and personalised care.
- Content is clinically reasonable: lifestyle changes (DASH diet, sodium limits, exercise, weight, stress, smoking) and seven drug classes with example drugs.
- It remains **generic and unsourced**: no BP thresholds for starting treatment, no first-line choice by patient group (e.g., age, diabetes, kidney disease), and nothing traceable to the Merck Manual.
- This is the "before" baseline for RAG: next, the same model receives the manual's own text as context, so answers can be grounded in a trusted source instead of training memory.

The above response is somewhat generic and is solely based on the data the model was trained on, rather than the medical manual.

Let's now provide our own context.

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [39]:
# Write code to define:
#   - qna_system_message: the system prompt describing the assistant's role, response guidelines, and fallback behavior
#   - qna_user_message_template: the user message template, including the ###Context and ###Question sections

# ==============================================================
# CELL R11: RAG SYSTEM MESSAGE AND USER MESSAGE TEMPLATE
# ==============================================================

# --- System message: the assistant's role and rules for using the manual ---
# Key difference from the earlier system_prompt: the model must now answer
# ONLY from the retrieved Merck Manual context, not from its own memory.
qna_system_message = """You are a clinical assistant for healthcare professionals.
You answer medical questions using ONLY the context provided from the Merck Manual.

The user message has two parts:
- ###Context: passages retrieved from the Merck Manual
- ###Question: the clinician's question

Rules:
1. Base every statement on the ###Context. Do not add facts from outside the context.
2. Structure the answer with clear headings and bullet points, in the order a clinician
   would act (assessment -> management -> treatment -> follow-up) where it applies.
3. Answer every part of the question. If the context covers only some parts,
   answer those and clearly state which parts are not covered.
4. If the context does not contain the answer, reply exactly:
   "I don't have enough information in the provided manual to answer this question."
5. Be concise and precise; keep drug names, doses and time frames exactly as in the context."""

# --- User message template: {context} and {question} are placeholders ---
# They get filled in for every question with .format(context=..., question=...).
qna_user_message_template = """
###Context
Here are some passages from the Merck Manual that are relevant to the question:
{context}

###Question
{question}
"""

print("RAG prompt template defined.")

RAG prompt template defined.


### Response Function

In [40]:
# Write a function called generate_rag_response() that:
#   - takes a user_input query and generation parameters (k, max_tokens, temperature, top_p) as input
#   - retrieves the top-k relevant document chunks for the query using the retriever
#   - combines the retrieved chunks into a single context string
#   - fills the qna_user_message_template with this context and the question
#   - calls the OpenAI client using the llm model, passing qna_system_message and the filled user message
#   - returns only the generated text of the response (wrap the call in a try/except so errors are returned as a message rather than raised)

# ==============================================================
# CELL R12: generate_rag_response() - THE CORE RAG FUNCTION
# Retrieve -> build context -> fill template -> ask the LLM
# ==============================================================

def generate_rag_response(user_input, k=3, max_tokens=800, temperature=0, top_p=0.95):
    """
    Answer a question using passages retrieved from the Merck Manual.

    Parameters
    ----------
    user_input  : the clinician's question
    k           : how many chunks to retrieve from the vector database
    max_tokens  : upper limit on answer length
    temperature : randomness (0 = focused and repeatable)
    top_p       : nucleus sampling threshold
    """

    # --- Step 1: Retrieve the k most relevant chunks ---
    # A fresh retriever is built each call so k can be changed in the fine-tuning section.
    rag_retriever = vectorstore.as_retriever(search_type="similarity",
                                             search_kwargs={"k": k})
    relevant_chunks = rag_retriever.invoke(user_input)

    # --- Step 2: Combine the chunks into one context string ---
    # Each chunk is labelled with its page number so answers stay traceable to the source.
    context_parts = []
    for doc in relevant_chunks:
        page_number = doc.metadata["page"] + 1          # metadata is 0-based, the PDF is 1-based
        context_parts.append(f"[Merck Manual, page {page_number}]\n{doc.page_content}")
    context_for_query = "\n\n".join(context_parts)      # blank line between chunks

    # --- Step 3: Fill the user message template with the context and the question ---
    user_message = qna_user_message_template.format(context=context_for_query,
                                                    question=user_input)

    # --- Step 4: Ask the LLM, with a safety net for API errors ---
    # try/except returns the error as text instead of crashing the whole notebook.
    try:
        completion = client.chat.completions.create(
            model=llm,
            messages=[
                {"role": "system", "content": qna_system_message},   # rules
                {"role": "user", "content": user_message},           # context + question
            ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        answer = completion.choices[0].message.content
    except Exception as error:
        answer = f"Sorry, I encountered the following error:\n{error}"

    return answer


print("generate_rag_response() is ready.")

generate_rag_response() is ready.


## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [41]:
# Write code to define the query above as user_input_1 and pass it to the generate_rag_response() function

# ==============================================================
# CELL R13: QUERY 1 - SEPSIS (RAG, default settings)
# ==============================================================

# --- Define the question exactly as given in the problem statement ---
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"

# --- Ask the RAG system (default: k=3, max_tokens=800, temperature=0, top_p=0.95) ---
# The answer is now built from Merck Manual passages, not the model's memory.
rag_answer_1 = generate_rag_response(user_input_1)

# --- Show the formatted answer ---
display(Markdown(rag_answer_1))

### Assessment
- Identify the presence of sepsis in the critically ill patient.
- Monitor vital signs, fluid intake and output, daily weight, and perform routine daily blood tests to detect problems early.

### Management
- Initiate broad-spectrum antibiotics as initial therapy:
  - For suspected methicillin-resistant S. aureus, use vancomycin plus an aminoglycoside.
  - If P. aeruginosa is prevalent, consider ceftazidime instead of an aminoglycoside.
  - For late-onset sepsis with coagulase-negative staphylococci, use vancomycin or replace it with nafcillin or cefazolin if the organism is sensitive.
- Remove any indwelling intravascular catheters if they are suspected to be the source of infection.

### Treatment
- Administer fluid resuscitation with 0.9% normal saline until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.
- Provide oxygen support.
- Drain abscesses and excise necrotic tissue as necessary.
- Normalize blood glucose levels.
- Consider replacement-dose corticosteroids.
- If hypotension persists after achieving target CVP or PAOP, administer dopamine to increase mean BP to at least 60 mm Hg. If dopamine exceeds 20 μg/kg/min, add norepinephrine.

### Follow-up
- Monitor systemic pressure, CVP, PAOP, pulse oximetry, ABGs, blood glucose, lactate, electrolyte levels, renal function, and urine output frequently.
- Adjust treatment based on culture results and ongoing assessment of the patient's condition. 

This protocol outlines the management of sepsis in a critical care unit based on the provided context.

**Observations:**
- **RAG filled the gaps both earlier answers missed**: *normalize blood glucose* and *replacement-dose corticosteroids* now appear, taken directly from the retrieved Merck Manual chunk (page 2456). This is grounding working as intended.
- It includes manual-specific targets not seen before: fluid resuscitation with 0.9% saline to **CVP 8 mm Hg or PAOP 12–15 mm Hg**, and vasopressor escalation to keep mean BP ≥ 60 mm Hg, plus a detailed monitoring list (CVP, PAOP, ABGs, lactate, electrolytes, renal function, urine output).
- **Retrieval noise leaked into the answer**: the antibiotic section (vancomycin + aminoglycoside, ceftazidime, coagulase-negative staphylococci, nafcillin/cefazolin) comes from the **neonatal sepsis** chunk (page 2996), which is the wrong patient group for an adult ICU question. RAG is only as good as what it retrieves.
- The model faithfully reflects the manual's approach (e.g., **dopamine before norepinephrine**, CVP-guided fluids), which differs from current guidelines (norepinephrine first-line, MAP ≥ 65 mm Hg). Grounded answers are **traceable but

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [42]:
# Write code to define the query above as user_input_2 and pass it to the generate_rag_response() function

# ==============================================================
# CELL R14: QUERY 2 - APPENDICITIS (RAG, default settings)
# ==============================================================

# --- Define the question exactly as given in the problem statement ---
user_input_2 = ("What are the common symptoms for appendicitis, and can it be cured via medicine? "
                "If not, what surgical procedure should be followed to treat it?")

# --- Ask the RAG system (default: k=3, max_tokens=800, temperature=0, top_p=0.95) ---
rag_answer_2 = generate_rag_response(user_input_2)

# --- Show the formatted answer ---
display(Markdown(rag_answer_2))

### Assessment
- **Common Symptoms of Appendicitis:**
  - Abdominal pain
  - Anorexia
  - Abdominal tenderness

### Management
- **Diagnosis:**
  - Clinical evaluation, often supplemented by CT or ultrasound.

### Treatment
- **Surgical Procedure:**
  - Treatment of acute appendicitis is surgical removal, specifically:
    - Open or laparoscopic appendectomy.
  - If the appendix is perforated, immediate surgery is necessary to prevent complications such as peritonitis.

### Follow-up
- **Post-Surgery Care:**
  - If the appendix is perforated, IV antibiotics should be continued until the patient's temperature and WBC count normalize or for a fixed course as per the surgeon's preference.
  
### Additional Information
- Appendicitis cannot be cured via medicine alone; surgical intervention is required for treatment.

**Observations:**
- The RAG answer is **concise and tightly grounded**: it states only what the retrieved passages support. Symptoms are limited to **abdominal pain, anorexia and tenderness**, fewer than the base answer, which drew on general training knowledge.
- It adds manual-specific details not seen before: **CT or ultrasound** to support diagnosis, **immediate surgery for a perforated appendix** to prevent peritonitis, and **IV antibiotics after perforation continued until temperature and WBC count normalise** (or for a fixed course).
- It answers all three parts and, like the earlier answers, concludes that appendicitis **cannot be cured by medicine alone**; open or laparoscopic appendectomy is the treatment. Antibiotic-only management for uncomplicated cases is not mentioned, so it either isn't in the retrieved chunks or isn't in this edition of the manual.
- The narrower symptom list suggests `k=3` may not retrieve the full clinical-features passage; a larger `k` could give a more complete picture and will be tested in fine-tuning.
- Trade-off: **less breadth than the base LLM, but every statement is traceable to the Merck Manual**, which is the priority for clinical reliability.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [43]:
# Write code to define the query above as user_input_3 and pass it to the generate_rag_response() function

# ==============================================================
# CELL R15: QUERY 3 - SUDDEN PATCHY HAIR LOSS (RAG, default settings)
# ==============================================================

# --- Define the question exactly as given in the problem statement ---
user_input_3 = ("What are the effective treatments or solutions for addressing sudden patchy hair loss, "
                "commonly seen as localized bald spots on the scalp, and what could be the possible "
                "causes behind it?")

# --- Ask the RAG system (default: k=3, max_tokens=800, temperature=0, top_p=0.95) ---
rag_answer_3 = generate_rag_response(user_input_3)

# --- Show the formatted answer ---
display(Markdown(rag_answer_3))

### Assessment
- **Condition**: Sudden patchy hair loss is indicative of alopecia areata.
- **Characteristics**: It presents as localized bald spots on the scalp and may affect other hairy areas, potentially leading to alopecia universalis.
- **Underlying Cause**: Alopecia areata is thought to be an autoimmune disorder affecting genetically susceptible individuals, triggered by unclear environmental factors.

### Management
- **Diagnosis**: A thorough evaluation may be necessary to confirm the diagnosis of alopecia areata and rule out other causes of hair loss.

### Treatment
- **Options for Alopecia Areata**:
  - **Topical Corticosteroids**: Applied directly to the affected areas.
  - **Intralesional Corticosteroids**: Injected into the bald spots.
  - **Systemic Corticosteroids**: For severe cases.
  - **Topical Minoxidil**: May be used, although its efficacy is not as well established for alopecia areata.
  - **Topical Anthralin**: A treatment option for localized areas.
  - **Topical Immunotherapy**: Such as diphencyprone or squaric acid dibutylester.
  - **Psoralen plus Ultraviolet A (PUVA)**: Another treatment modality for severe cases.

### Follow-Up
- **Monitoring**: Regular follow-up is necessary to assess the response to treatment and make adjustments as needed.

### Summary of Causes
- **Possible Causes**: The exact environmental triggers for alopecia areata remain unclear, but it is recognized as an autoimmune condition. 

This information provides a comprehensive approach to managing sudden patchy hair loss due to alopecia areata.

**Observations:**
- RAG gave the **most accurate account of the cause** so far: alopecia areata is an **autoimmune disorder in genetically susceptible people, triggered by unclear environmental factors**. It no longer mixes in look-alikes such as tinea capitis or hormonal changes, which the base answer wrongly listed as causes.
- It adds manual-specific details: possible progression to **alopecia universalis**, **squaric acid dibutylester** as a second immunotherapy agent, and **PUVA** for severe cases. It is also honest that **minoxidil's efficacy is not well established** for alopecia areata, a nuance the earlier answers missed.
- The treatment ladder matches the manual: topical → intralesional → systemic corticosteroids, plus anthralin and topical immunotherapy (diphencyprone).
- It is **thinner on practical detail**: no doses, no treatment ordering by severity, and "rule out other causes" is stated without naming them (e.g., tinea capitis, trichotillomania).
- Grounding improved **correctness** at some cost to **breadth**; a larger `k` could pull in the differential-diagnosis and dosing passages.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [44]:
# Write code to define the query above as user_input_4 and pass it to the generate_rag_response() function

# ==============================================================
# CELL R16: QUERY 4 - TRAUMATIC BRAIN INJURY (RAG, default settings)
# ==============================================================

# --- Define the question exactly as given in the problem statement ---
user_input_4 = ("What treatments are recommended for a person who has sustained a physical injury "
                "to brain tissue, resulting in temporary or permanent impairment of brain function?")

# --- Ask the RAG system (default: k=3, max_tokens=800, temperature=0, top_p=0.95) ---
rag_answer_4 = generate_rag_response(user_input_4)

# --- Show the formatted answer ---
display(Markdown(rag_answer_4))

### Assessment
- Diagnosis of traumatic brain injury (TBI) is suspected clinically and confirmed by imaging (primarily CT).
- Initial rapid neurologic assessment includes recording GCS, pupillary reaction, BP, pulse, and temperature frequently for several hours.

### Management
- Ensure a reliable airway.
- Maintain adequate ventilation, oxygenation, and blood pressure.
- Monitor for deterioration in neurologic findings, which demands prompt attention.

### Treatment
- **Mild Injuries**: Discharge and observation.
- **Moderate and Severe Injuries**:
  - Optimization of ventilation, oxygenation, and brain perfusion.
  - Treatment of complications such as increased intracranial pressure (ICP), seizures, and hematomas.
  - Surgical intervention may be necessary to place monitors for ICP, decompress the brain, or remove hematomas.
  - Pain relief with short-acting opioids (e.g., fentanyl) as needed.

### Follow-Up
- Many patients require rehabilitation after the initial treatment phase.
- Continuous monitoring for cognitive and neuropsychiatric deficits, which may persist after severe and moderate TBI.

This summary outlines the recommended treatments for individuals with TBI as per the provided context.

**Observations:**
- **Safe and grounded**: unlike the base LLM, the RAG answer does **not** recommend corticosteroids. Every recommendation comes from the retrieved Merck Manual passages.
- It is the first answer to **stratify treatment by severity**: mild injuries → discharge and observation; moderate/severe → optimise ventilation, oxygenation and brain perfusion, treat raised ICP, seizures and hematomas, and surgery for ICP monitoring, decompression or hematoma removal.
- It adds manual-specific assessment details: diagnosis confirmed **primarily by CT**, and frequent recording of **GCS, pupillary reaction, BP, pulse and temperature** for several hours. It also includes pain control with **short-acting opioids (e.g., fentanyl)** and follow-up for persistent cognitive and neuropsychiatric deficits.
- It is **less specific on medical therapy** than the prompt-engineered answer: no mention of mannitol/hypertonic saline, seizure-prophylaxis drugs or numeric targets, and rehabilitation is only named, not described. These details likely sit in chunks outside the top 3.
- Overall, RAG delivered the **most trustworthy** TBI answer (safe, source-based and severity-guided) at the cost of some depth, which a larger `k` may recover.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [45]:
# Write code to define the query above as user_input_5 and pass it to the generate_rag_response() function

# ==============================================================
# CELL R17: QUERY 5 - LEG FRACTURE ON A HIKING TRIP (RAG, default settings)
# ==============================================================

# --- Define the question exactly as given in the problem statement ---
user_input_5 = ("What are the necessary precautions and treatment steps for a person who has fractured "
                "their leg during a hiking trip, and what should be considered for their care and recovery?")

# --- Ask the RAG system (default: k=3, max_tokens=800, temperature=0, top_p=0.95) ---
rag_answer_5 = generate_rag_response(user_input_5)

# --- Show the formatted answer ---
display(Markdown(rag_answer_5))

### Assessment
- **Symptoms**: Gradual onset of pain, swelling, and localized bone tenderness may indicate a fracture.
- **Imaging**: Plain x-rays should be performed to confirm the fracture, although they may not show the fracture initially. A repeat x-ray may be necessary in 2 to 3 weeks to check for callus formation.

### Management
- **Immediate Care**: 
  - Rest the affected leg.
  - Elevate the leg to reduce swelling.
  - Apply ice to the area to manage pain and swelling.
  - Consider analgesics for pain relief.
  
- **Immobilization**: 
  - Use a splint for stabilization, allowing for some movement and ice application without contributing to compartment syndrome.
  - If the fracture is stable, a splint can be used for several days or less.

### Treatment
- **Fracture Treatment**: 
  - For stress fractures, treatment includes rest, elevation, analgesics, and sometimes immobilization.
  - Early mobilization may be beneficial for rapidly healing injuries to minimize contractures and muscle atrophy.

### Follow-Up
- **Monitoring**: 
  - Patients should seek medical care if there is an odor from the cast or if fever develops, as these may indicate infection.
  - Regular follow-up appointments to monitor healing and adjust treatment as necessary.

### Considerations for Care and Recovery
- **Elderly Patients**: 
  - Be aware that elderly patients are at higher risk for complications such as deep venous thrombosis and muscle atrophy due to prolonged immobilization.
  - Early mobilization and physical therapy are essential for recovery of function, especially in older adults.

- **Complications**: 
  - Monitor for signs of compartment syndrome and other complications such as joint contractures and pressure ulcers.

This structured approach ensures comprehensive care for a leg fracture sustained during a hiking trip.

**Observations:**
- **Retrieval picked the wrong kind of fracture**: the answer describes *gradual onset* of pain and treatment "for stress fractures", but the question is about an **acute traumatic fracture on a hike**. The similarity search matched "fracture + leg" to a stress-fracture passage, a clear retrieval miss.
- As a result, the **emergency, on-trail steps are missing**: no splinting before moving, no open-fracture wound care, no checking circulation, and no evacuation or calling for help. These were all covered by the base and prompt-engineered answers.
- It does add useful, manual-grounded recovery points: a **repeat X-ray in 2–3 weeks** because early films may miss the fracture, splints that avoid causing **compartment syndrome**, early mobilisation to prevent **contractures and muscle atrophy**, and higher **DVT and pressure-ulcer risk in elderly patients**.
- This shows RAG's key weakness: when the top chunks are **semantically close but contextually wrong**, the model faithfully answers the wrong question. Better retrieval (larger `k`, query rewording, or MMR search) is needed, and that's tested next in fine-tuning.

### Fine-tuning

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [47]:
# Write code to call generate_rag_response() for the query above with a different temperature value (e.g. 0.5)

# ==============================================================
# CELL R18: FINE-TUNING 1 - SEPSIS with higher temperature
# Change ONE setting vs the default RAG run: temperature 0 -> 0.5
# ==============================================================

# --- Ask the RAG system with more randomness ---
# Everything else stays at the defaults (k=3, max_tokens=800, top_p=0.95),
# so any difference in the answer comes from temperature alone.
ft_answer_1 = generate_rag_response(user_input_1, temperature=0.5)

# --- Show the formatted answer ---
display(Markdown(ft_answer_1))

### Assessment
- Monitor vital signs (temperature, BP, pulse, respiration rate).
- Quantify fluid intake and output; daily weight measurement.
- Conduct routine daily blood tests to detect problems early.
- Monitor systemic pressure, CVP, PAOP, pulse oximetry, ABGs, blood glucose, lactate, and electrolyte levels.
- Assess renal function and urine output, using an indwelling catheter if necessary.

### Management
- Initiate fluid resuscitation with 0.9% normal saline until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.
- Administer broad-spectrum antibiotics, modified according to culture results.
- Provide oxygen therapy.
- Drain abscesses and excise necrotic tissue as needed.
- Normalize blood glucose levels.
- Consider replacement-dose corticosteroids.

### Treatment
- For hypotension that persists after achieving target CVP or PAOP, administer dopamine to increase mean BP to at least 60 mm Hg. 
- If dopamine exceeds 20 μg/kg/min, add norepinephrine as a second vasopressor.

### Follow-Up
- Continuously monitor the patient in the ICU for response to treatment and any complications.
- Adjust treatment based on ongoing assessments and lab results.

**Observations:**
- **Temperature 0 → 0.5** (k=3, top_p=0.95, max_tokens=800 unchanged): the **core clinical facts stayed identical** to the default RAG answer: 0.9% saline to **CVP 8 mm Hg / PAOP 12–15 mm Hg**, broad-spectrum antibiotics adjusted to cultures, O₂, source control, **glucose normalisation**, **replacement-dose corticosteroids**, and dopamine → norepinephrine escalation. With grounded context, higher temperature changed **wording and organisation, not the facts**.
- The **neonatal antibiotic details** (vancomycin + aminoglycoside, ceftazidime) that leaked in at temperature 0 were **not used** this time; the answer drew only on the adult septic-shock passage, which made it more relevant for an adult ICU.
- This isn't a reliable improvement: the same noisy chunk was still retrieved, and at higher temperature the model simply chose differently. For clinical use, **repeatability matters more**, so a low temperature (0–0.2) is still preferred.
- Assessment is more detailed (fluid balance, daily weights, CVP/PAOP, ABGs, lactate, urine output via catheter), but antibiotics are now generic, so the answer lost specificity in exchange for relevance.

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [48]:
# Write code to call generate_rag_response() for the query above with a different top_p value

# ==============================================================
# CELL R19: FINE-TUNING 2 - APPENDICITIS with lower top_p
# Change ONE setting vs the default RAG run: top_p 0.95 -> 0.5
# ==============================================================

# --- Ask the RAG system with a narrower word pool ---
# top_p=0.5 means the model only picks from the words that make up the top 50%
# of probability: more conservative, less varied wording.
# Temperature is set to 0.5 so top_p actually has an effect (at temperature 0 the
# model always picks the single most likely word, which makes top_p irrelevant).
ft_answer_2 = generate_rag_response(user_input_2, temperature=0.5, top_p=0.5)

# --- Show the formatted answer ---
display(Markdown(ft_answer_2))

### Assessment
- **Common Symptoms of Appendicitis:**
  - Abdominal pain
  - Anorexia
  - Abdominal tenderness

### Management
- **Diagnosis:**
  - Clinical diagnosis, often supplemented by CT or ultrasound.

### Treatment
- **Surgical Procedure:**
  - Treatment of acute appendicitis is surgical removal, specifically:
    - Open or laparoscopic appendectomy.
  - If the appendix is perforated, immediate surgery is necessary to prevent complications such as peritonitis.

### Follow-up
- **Post-Surgery Care:**
  - For nonperforated appendicitis, no further antibiotics are required after surgery.
  - If the appendix is perforated, antibiotics should be continued until the patient's temperature and WBC count have normalized or for a fixed course as per the surgeon's preference.

### Conclusion
- Appendicitis cannot be cured via medicine alone; surgical intervention is necessary.

**Observations:**
- **top_p 0.95 → 0.5, with temperature 0.5** (k=3, max_tokens=800): top_p only has an effect when temperature > 0, so both were changed together. Restricting the model to the most likely words produced an answer **nearly identical** to the default RAG run: same symptoms (abdominal pain, anorexia, tenderness), CT/ultrasound, open or laparoscopic appendectomy, and immediate surgery if perforated.
- It added one useful, manual-grounded detail: **no further antibiotics are needed after surgery for non-perforated appendicitis**, while perforated cases continue antibiotics until temperature and WBC normalise.
- The low top_p **cancelled out most of the randomness from temperature 0.5**, confirming top_p acts as a guard-rail: output stays conservative and consistent, which suits clinical use.
- Since the answer barely changed, the **limiting factor is retrieval, not generation**: the symptom list stays short because the top-3 chunks contain only those symptoms. Tuning `k` (tested in Query 5) is more likely to add content than tuning top_p.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [51]:
# Write code to call generate_rag_response() for the query above with a different max_tokens value

# ==============================================================
# CELL R20: FINE-TUNING 3 - HAIR LOSS with a smaller max_tokens
# Change ONE setting vs the default RAG run: max_tokens 800 -> 300
# ==============================================================

# --- Ask the RAG system with a much tighter length limit ---
# Tests whether the answer stays complete and useful when forced to be short,
# or whether it gets cut off mid-sentence.
ft_answer_3 = generate_rag_response(user_input_3, max_tokens=300)

# --- Show the formatted answer ---
display(Markdown(ft_answer_3))

### Assessment
- **Condition**: Sudden patchy hair loss is indicative of alopecia areata.
- **Characteristics**: It presents as sudden patchy hair loss, often affecting the scalp and beard, and can involve any hairy area, potentially leading to alopecia universalis.
- **Underlying Causes**: Alopecia areata is thought to be an autoimmune disorder affecting genetically susceptible individuals, triggered by unclear environmental factors.

### Management
- **Diagnosis**: A thorough evaluation may be necessary to rule out other skin or systemic disorders.

### Treatment
- **Options for Alopecia Areata**:
  - **Topical Corticosteroids**: For localized treatment.
  - **Intralesional Corticosteroids**: For more severe or widespread cases.
  - **Systemic Corticosteroids**: In severe cases.
  - **Topical Minoxidil**: May be used, although its effectiveness is not guaranteed.
  - **Topical Anthralin**: Another treatment option.
  - **Topical Immunotherapy**: Such as diphencyprone or squaric acid dibutylester.
  - **Psoralen plus Ultraviolet A (PUVA)**: For severe cases.

### Follow-Up
- **Monitoring**: Efficacy of treatment is usually evident within 6 to 8 months, and follow-up should assess hair regrowth and any adverse effects from treatments.

### Summary
- **Effective Treatments**

**Observations:**
- **max_tokens 800 → 300** (k=3, temperature=0, top_p=0.95): even with less than half the token budget, the answer kept **all the key content** of the default RAG run: alopecia areata as an autoimmune condition, the full treatment ladder (topical, intralesional and systemic corticosteroids, minoxidil, anthralin, diphencyprone/squaric acid, PUVA) and follow-up.
- It added one new, useful manual detail: **treatment efficacy is usually evident within 6–8 months**, which helps set patient expectations.
- The model **compressed each point to a short phrase** to fit the budget.
- The answer was **cut off in the final Summary** (it stops at a heading with no content): the limit truncates output **without any warning**, so 300 tokens is too tight for a multi-part clinical question. A budget of 500–800 is safer.

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [50]:
# Write code to call generate_rag_response() for the query above with a different top_p and temperature value

# ==============================================================
# CELL R21: FINE-TUNING 4 - BRAIN INJURY with higher temperature AND top_p
# Change TWO settings vs the default RAG run: temperature 0 -> 0.7, top_p 0.95 -> 1.0
# ==============================================================

# --- Ask the RAG system at the most "creative" settings tested so far ---
# temperature=0.7 adds randomness; top_p=1.0 lets the model consider its full vocabulary.
# Tests whether the grounded context keeps the answer safe and factual under high randomness.
ft_answer_4 = generate_rag_response(user_input_4, temperature=0.7, top_p=1.0)

# --- Show the formatted answer ---
display(Markdown(ft_answer_4))

### Assessment
- Diagnosis of traumatic brain injury (TBI) is suspected clinically and confirmed by imaging (primarily CT).
- Initial evaluation includes:
  - Neurologic findings (GCS and pupillary reaction)
  - Blood pressure, pulse, and temperature monitoring.

### Management
- Ensure a reliable airway.
- Maintain adequate ventilation, oxygenation, and blood pressure.
- Monitor for any deterioration in neurologic status that demands prompt attention.

### Treatment
- For mild injuries:
  - Discharge and observation.
  
- For moderate and severe injuries:
  - Optimization of ventilation, oxygenation, and brain perfusion.
  - Treatment of complications such as increased ICP, seizures, and hematomas.
  - Surgical intervention may be necessary for severe injuries to monitor and treat intracranial pressure, decompress the brain, or remove hematomas.

### Follow-up
- Rehabilitation may be required for many patients after initial treatment and stabilization. 

This summarizes the recommended treatments for individuals with TBI as outlined in the provided context.

**Observations:**
- **temperature 0 → 0.7 and top_p 0.95 → 1.0** (k=3, max_tokens=800), the most random settings tested: the answer stayed **safe and factually consistent** with the default RAG run. No corticosteroids or other unsupported advice appeared, and the severity-based structure was kept (mild → discharge and observation; moderate/severe → ventilation, oxygenation, perfusion, ICP/seizure/hematoma management, surgery).
- The grounded context acted as an **anchor**: even with high randomness, the model had only the manual's content to draw on, so variation showed up in wording, not facts. That's a key advantage of RAG over the base LLM, which gave unsafe advice even at temperature 0.
- The answer was **slightly shorter and less specific**: the fentanyl pain-control detail and the note on persistent neuropsychiatric deficits from the default run were dropped. Higher randomness produced a less complete answer.
- Conclusion: RAG is robust to parameter changes for safety, but **low temperature (0–0.2) still gives the most complete, repeatable answers** for clinical use.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [52]:
# Write code to call generate_rag_response() for the query above with a different k value

# ==============================================================
# CELL R22: FINE-TUNING 5 - LEG FRACTURE with more retrieved chunks
# Change ONE setting vs the default RAG run: k 3 -> 6
# ==============================================================

# --- Ask the RAG system with twice as many chunks from the manual ---
# The default k=3 retrieved stress-fracture passages instead of acute trauma care.
# Retrieving 6 chunks gives the model a wider slice of the manual to work from.
ft_answer_5 = generate_rag_response(user_input_5, k=6)

# --- Show the formatted answer ---
display(Markdown(ft_answer_5))

### Assessment
- **Injury Type**: Confirm if the fracture is open or closed.
- **Symptoms**: Look for signs of compartment syndrome (e.g., severe pain, swelling, numbness).
- **Complications**: Assess for potential complications such as rapid blood loss or fat embolism.

### Management
- **Immediate Care**: 
  - Move the patient to a safe location away from potential hazards.
  - Immobilize the leg in a functional position just below heart level.
  - Remove any constrictive clothing or jewelry.

### Treatment
- **Analgesia**: Administer pain relief as needed.
- **Splinting**: Apply a splint to stabilize the fracture.
- **Wound Care**: If the fracture is open, apply sterile dressings and consider tetanus prophylaxis and broad-spectrum antibiotics (e.g., a 2nd-generation cephalosporin plus an aminoglycoside).
- **Transport**: Ensure rapid transport to a medical facility for further evaluation and treatment.

### Follow-Up
- **Monitoring**: After initial treatment, monitor for signs of complications such as infection or compartment syndrome.
- **Rehabilitation**: Plan for early mobilization and physical therapy to promote recovery and prevent stiffness, contractures, and muscle atrophy.
- **Considerations for Recovery**: Take into account the patient's age and any coexisting disorders (e.g., arthritis) that may affect recovery.

### Additional Considerations
- **Hygiene**: Emphasize the importance of good hygiene to prevent infection.
- **Long-term Care**: Be aware that prolonged immobilization can lead to complications, especially in elderly patients.

**Observations:**
- **k 3 → 6** (temperature=0, top_p=0.95, max_tokens=800): doubling the retrieved chunks **fixed the retrieval miss**. The default run answered about *stress fractures*; with k=6 the answer now covers an **acute traumatic fracture**, which is what the question asks.
- New, manual-grounded emergency content appears: checking **open vs closed** fracture, watching for **compartment syndrome, rapid blood loss and fat embolism**, immobilising the leg and **splinting**, removing constrictive clothing/jewellery, open-fracture care (**sterile dressing, tetanus prophylaxis, broad-spectrum antibiotics**), and rapid transport.
- It still lists "move the patient to a safe location" before immobilisation; standard first aid is to splint first and move only if there's immediate danger. Hiking-specific points (calling for help, evacuation) remain limited.
- Recovery points are kept: early mobilisation and physiotherapy to prevent stiffness, contractures and atrophy, with extra caution in elderly patients.
- **Key finding across all fine-tuning runs: `k` had the biggest impact on answer quality.** Temperature, top_p and max_tokens mainly changed wording and length, while `k` changed *what information the model had*. For multi-part clinical questions, **k=5–6** is a better default than k=3.

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answers generated to the questions from the previous section.

- We use a **different model as the judge** (`judge_llm`) from the one used for answer generation (`llm`), via the GL Proxy. This avoids the generator model rating its own output, which can otherwise inflate scores due to self-preference bias.

In [53]:
# Write code to define groundedness_rater_system_message: a system prompt instructing the judge model to
# rate, on a scale of 1-5, whether the answer is derived only from the given context.
# Ask the model to return only the final score, in a dictionary format, e.g. {groundedness_score:4}

# ==============================================================
# CELL E1: GROUNDEDNESS JUDGE PROMPT
# Does the answer come ONLY from the retrieved context? (checks for hallucination)
# ==============================================================

groundedness_rater_system_message = """You are a strict evaluator of a medical question-answering system.

You will receive three sections:
- ###Question: the user's question
- ###Context: passages retrieved from the Merck Manual
- ###Answer: the answer generated by the AI

Your task: rate how well the ###Answer is GROUNDED in the ###Context,
i.e. whether every statement in the answer is supported by the context.

Scoring scale (1-5):
5 - Every statement is directly supported by the context.
4 - Almost all statements are supported; minor details are not explicitly in the context.
3 - Some statements are supported, but noticeable parts are not found in the context.
2 - Most statements are not supported by the context.
1 - The answer is not based on the context at all.

Judge ONLY against the context, not your own medical knowledge.
Return ONLY the final score in this exact dictionary format, with no explanation:
{groundedness_score: <score>}"""

print("Groundedness judge prompt defined.")

Groundedness judge prompt defined.


In [59]:
# Write code to define relevance_rater_system_message: a system prompt instructing the judge model to
# rate, on a scale of 1-5, how well the answer addresses the main aspects of the question given the context.
# Ask the model to return only the final score, in a dictionary format, e.g. {relevance_score:4}

# ==============================================================
# CELL E2: RELEVANCE JUDGE PROMPT
# Does the answer actually address what was asked? (checks completeness and focus)
# ==============================================================

relevance_rater_system_message = """You are a strict evaluator of a medical question-answering system.

You will receive three sections:
- ###Question: the user's question
- ###Context: passages retrieved from the Merck Manual
- ###Answer: the answer generated by the AI

Your task: rate how RELEVANT the ###Answer is to the ###Question,
i.e. how well it addresses the main aspects of the question, given the context.

Scoring scale (1-5):
5 - Fully addresses every part of the question, clearly and on topic.
4 - Addresses the main parts; minor aspects are missing or thin.
3 - Partially addresses the question; one or more important parts are missing.
2 - Mostly off-topic or misses most of what was asked.
1 - Does not address the question at all.

Pay special attention to multi-part questions: each part must be answered.
Return ONLY the final score in this exact dictionary format, with no explanation:
{relevance_score: <score>}"""

print("Relevance judge prompt defined.")

Relevance judge prompt defined.


In [60]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [61]:
# Write a function called generate_ground_relevance_response() that:
#   - takes a user_input query and generation parameters (k, max_tokens, temperature, top_p) as input
#   - retrieves the relevant document chunks and combines them into a context, as in generate_rag_response()
#   - generates an answer using the GENERATION model (llm), via the OpenAI client
#   - fills user_message_template with the context, question, and generated answer
#   - sends this filled prompt to the JUDGE model (judge_llm) twice: once with groundedness_rater_system_message,
#     and once with relevance_rater_system_message, as the system message
#   - returns both the groundedness score response and the relevance score response

# ==============================================================
# CELL E3: generate_ground_relevance_response() - LLM-AS-A-JUDGE
# Generate a RAG answer with llm, then have judge_llm score it twice
# ==============================================================

def generate_ground_relevance_response(user_input, k=3, max_tokens=800, temperature=0, top_p=0.95):
    """
    Answer a question with RAG, then score the answer for groundedness and relevance.
    Returns: (groundedness_result, relevance_result, answer)
    """

    # --- Step 1: Retrieve the k most relevant chunks (same as generate_rag_response) ---
    rag_retriever = vectorstore.as_retriever(search_type="similarity",
                                             search_kwargs={"k": k})
    relevant_chunks = rag_retriever.invoke(user_input)

    context_parts = []
    for doc in relevant_chunks:
        page_number = doc.metadata["page"] + 1
        context_parts.append(f"[Merck Manual, page {page_number}]\n{doc.page_content}")
    context_for_query = "\n\n".join(context_parts)

    # --- Step 2: Generate the answer with the GENERATION model (llm) ---
    qna_message = qna_user_message_template.format(context=context_for_query,
                                                   question=user_input)
    answer_completion = client.chat.completions.create(
        model=llm,
        messages=[{"role": "system", "content": qna_system_message},
                  {"role": "user", "content": qna_message}],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    answer = answer_completion.choices[0].message.content

    # --- Step 3: Fill the judge template with question, context and answer ---
    judge_message = user_message_template.format(question=user_input,
                                                 context=context_for_query,
                                                 answer=answer)

    # --- Step 4: Ask the JUDGE model (judge_llm) to score groundedness ---
    # temperature=0 so the judge scores consistently every time.
    groundedness_completion = client.chat.completions.create(
        model=judge_llm,
        messages=[{"role": "system", "content": groundedness_rater_system_message},
                  {"role": "user", "content": judge_message}],
        max_tokens=50,
        temperature=0,
    )
    groundedness_result = groundedness_completion.choices[0].message.content

    # --- Step 5: Ask the JUDGE model to score relevance ---
    relevance_completion = client.chat.completions.create(
        model=judge_llm,
        messages=[{"role": "system", "content": relevance_rater_system_message},
                  {"role": "user", "content": judge_message}],
        max_tokens=50,
        temperature=0,
    )
    relevance_result = relevance_completion.choices[0].message.content

    return groundedness_result, relevance_result, answer


print("generate_ground_relevance_response() is ready.")

generate_ground_relevance_response() is ready.


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [62]:
# Write code to call generate_ground_relevance_response() for the query above (e.g. with max_tokens=370) and print both scores

# ==============================================================
# CELL E4: EVALUATE QUERY 1 - SEPSIS
# The template suggests max_tokens=370 for this one, so we use it here.
# ==============================================================

# --- Generate the RAG answer and get both judge scores ---
ground_1, relevance_1, eval_answer_1 = generate_ground_relevance_response(user_input_1, max_tokens=370)

# --- Show the scores first (the main result) ---
print("Groundedness:", ground_1)
print("Relevance   :", relevance_1)

# --- Show the answer that was scored, so the scores can be checked ---
print("\n" + "=" * 80)
print("ANSWER THAT WAS SCORED")
print("=" * 80)
display(Markdown(eval_answer_1))

Groundedness: {groundedness_score: 5}
Relevance   : {relevance_score: 5}

ANSWER THAT WAS SCORED


### Assessment
- Identify the presence of sepsis in the critically ill patient.
- Monitor vital signs, fluid intake and output, daily weight, and perform routine daily blood tests to detect problems early.

### Management
- Initiate broad-spectrum antibiotics as initial therapy:
  - For suspected methicillin-resistant S. aureus, use vancomycin plus an aminoglycoside.
  - If P. aeruginosa is prevalent, consider ceftazidime instead of an aminoglycoside.
  - For late-onset sepsis with coagulase-negative staphylococci, use vancomycin unless the organism is sensitive to nafcillin, in which case use nafcillin or cefazolin.
- Remove any indwelling intravascular catheters if they are suspected sources of infection.

### Treatment
- Administer fluid resuscitation with 0.9% normal saline until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.
- Provide oxygen support.
- Drain abscesses and excise necrotic tissue as necessary.
- Normalize blood glucose levels.
- Consider replacement-dose corticosteroids.
- Monitor systemic pressure, CVP, PAOP, pulse oximetry, ABGs, blood glucose, lactate, electrolyte levels, renal function, and urine output.

### Follow-up
- Continuously monitor the patient in the ICU for response to treatment and adjust management based on culture results and clinical status.
- If hypotension persists after achieving target CVP or PAOP, consider administering dopamine to increase mean BP to at least 60 mm Hg, and potentially add norepinephrine if dopamine doses exceed 20 μg/kg/min. 

This protocol outlines the management of sepsis in a critical care unit based on

**Observations:**
- **Groundedness 5/5, Relevance 5/5.** The judge (`gpt-4o`) confirmed every statement comes from the retrieved Merck Manual passages and that the answer covers the sepsis protocol end to end (assessment, antibiotics, fluids, O₂, source control, glucose, corticosteroids, vasopressors, monitoring).
- With `max_tokens=370`, the answer was **cut off at the very last sentence** ("...based on"). All clinical content was delivered, so it didn't affect the scores, but it confirms 370 tokens is at the limit for this question.
- **The judge has a blind spot**: the answer still includes **neonatal antibiotic regimens** (vancomycin + aminoglycoside, coagulase-negative staphylococci), which don't fit an adult ICU question. Groundedness is correctly 5 because that text *is* in the context, but relevance still scored 5, so the judge didn't penalise the wrong patient population.
- Takeaway: LLM-as-a-judge is a useful, fast quality check, but a perfect score doesn't guarantee clinical fit. **Human review is still needed** for high-stakes medical answers.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [63]:
# Write code to call generate_ground_relevance_response() for the query above and print both scores

# ==============================================================
# CELL E5: EVALUATE QUERY 2 - APPENDICITIS
# Default settings (k=3, max_tokens=800, temperature=0, top_p=0.95)
# ==============================================================

# --- Generate the RAG answer and get both judge scores ---
ground_2, relevance_2, eval_answer_2 = generate_ground_relevance_response(user_input_2)

# --- Show the scores first (the main result) ---
print("Groundedness:", ground_2)
print("Relevance   :", relevance_2)

# --- Show the answer that was scored ---
print("\n" + "=" * 80)
print("ANSWER THAT WAS SCORED")
print("=" * 80)
display(Markdown(eval_answer_2))

Groundedness: {groundedness_score: 5}
Relevance   : {relevance_score: 5}

ANSWER THAT WAS SCORED


### Assessment
- **Common Symptoms of Appendicitis:**
  - Abdominal pain
  - Anorexia
  - Abdominal tenderness

### Management
- **Diagnosis:**
  - Clinical diagnosis, often supplemented by CT or ultrasound.

### Treatment
- **Surgical Procedure:**
  - Treatment of acute appendicitis is surgical removal, specifically:
    - Open or laparoscopic appendectomy.
  - If the appendix is perforated, immediate surgery is necessary to prevent complications such as peritonitis.

### Follow-up
- **Post-Surgery Care:**
  - If the appendix is perforated, IV antibiotics should be continued until the patient's temperature and WBC count have normalized or for a fixed course as per the surgeon's preference.
  
### Additional Information
- Appendicitis cannot be cured via medicine alone; surgical intervention is required for treatment.

**Observations:**
- **Groundedness 5/5, Relevance 5/5.** The answer covers all three parts of the question (symptoms, whether medicine can cure it, and the surgical procedure), and every statement traces back to the retrieved Merck Manual passages.
- With temperature=0 the answer is **identical to the default RAG run**, which confirms the pipeline gives repeatable, consistent output.
- The high relevance score is justified, but the judge rates *coverage*, not *depth*: the symptom list is short (pain, anorexia, tenderness) and antibiotic-only treatment for uncomplicated cases isn't discussed. A clinician may still want more detail even with a perfect score.
- Both scores being 5 for a concise, source-bound answer shows the RAG system reliably avoids hallucination on well-covered topics.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [64]:
# Write code to call generate_ground_relevance_response() for the query above and print both scores

# ==============================================================
# CELL E6: EVALUATE QUERY 3 - SUDDEN PATCHY HAIR LOSS
# Default settings (k=3, max_tokens=800, temperature=0, top_p=0.95)
# ==============================================================

# --- Generate the RAG answer and get both judge scores ---
ground_3, relevance_3, eval_answer_3 = generate_ground_relevance_response(user_input_3)

# --- Show the scores first (the main result) ---
print("Groundedness:", ground_3)
print("Relevance   :", relevance_3)

# --- Show the answer that was scored ---
print("\n" + "=" * 80)
print("ANSWER THAT WAS SCORED")
print("=" * 80)
display(Markdown(eval_answer_3))

Groundedness: {groundedness_score: 5}
Relevance   : {relevance_score: 5}

ANSWER THAT WAS SCORED


### Assessment
- **Condition**: Sudden patchy hair loss is identified as alopecia areata.
- **Characteristics**: It presents as sudden patchy hair loss, often affecting the scalp and beard, and can involve any hairy area, potentially leading to alopecia universalis.
- **Underlying Cause**: Alopecia areata is thought to be an autoimmune disorder affecting genetically susceptible individuals, with unclear environmental triggers.

### Management
- **Diagnosis**: A thorough evaluation may be necessary to rule out other skin or systemic disorders.

### Treatment
- **Options for Alopecia Areata**:
  - **Topical Corticosteroids**: For localized treatment.
  - **Intralesional Corticosteroids**: For more severe or widespread cases.
  - **Systemic Corticosteroids**: In severe cases.
  - **Topical Minoxidil**: May be used, although its efficacy is not guaranteed.
  - **Topical Anthralin**: Another treatment option.
  - **Topical Immunotherapy**: Such as diphencyprone or squaric acid dibutylester.
  - **Psoralen plus Ultraviolet A (PUVA)**: For severe cases.

### Follow-Up
- **Monitoring**: Efficacy of treatment is usually evident within 6 to 8 months. Regular follow-up is necessary to assess response and adjust treatment as needed.

### Summary
- **Effective Treatments**: Include topical and intralesional corticosteroids, systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy, and PUVA.
- **Possible Causes**: Primarily an autoimmune disorder with genetic susceptibility and environmental triggers. 

No additional information on other causes of sudden patchy hair loss is provided in the context.

**Observations:**
- **Groundedness 5/5, Relevance 5/5.** The answer identifies alopecia areata as an autoimmune condition and lists the full treatment ladder from the manual (topical, intralesional and systemic corticosteroids, minoxidil, anthralin, immunotherapy, PUVA), with 6–8 month follow-up.
- The model **explicitly stated its limit**: *"No additional information on other causes of sudden patchy hair loss is provided in the context."* This is rule 3 of the RAG system prompt working, and it's safer than inventing causes, which the base LLM did (it listed tinea capitis and hormonal changes as causes).
- The perfect relevance score is slightly generous: the "possible causes" part of the question is only answered for alopecia areata, with no differential causes (tinea capitis, trichotillomania, traction alopecia).
- This shows the trade-off clearly: RAG **prefers honest gaps over confident guesses**. That's the right behaviour for a clinical tool, but retrieval coverage (larger `k`) determines how complete the answer can be.

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [65]:
# Write code to call generate_ground_relevance_response() for the query above and print both scores

# ==============================================================
# CELL E7: EVALUATE QUERY 4 - TRAUMATIC BRAIN INJURY
# Default settings (k=3, max_tokens=800, temperature=0, top_p=0.95)
# ==============================================================

# --- Generate the RAG answer and get both judge scores ---
ground_4, relevance_4, eval_answer_4 = generate_ground_relevance_response(user_input_4)

# --- Show the scores first (the main result) ---
print("Groundedness:", ground_4)
print("Relevance   :", relevance_4)

# --- Show the answer that was scored ---
print("\n" + "=" * 80)
print("ANSWER THAT WAS SCORED")
print("=" * 80)
display(Markdown(eval_answer_4))

Groundedness: {groundedness_score: 5}
Relevance   : {relevance_score: 5}

ANSWER THAT WAS SCORED


### Assessment
- Diagnosis of traumatic brain injury (TBI) is suspected clinically and confirmed by imaging (primarily CT).
- Initial rapid neurologic assessment includes recording GCS, pupillary reaction, BP, pulse, and temperature frequently for several hours.

### Management
- Ensure a reliable airway.
- Maintain adequate ventilation, oxygenation, and blood pressure.
- Monitor for deterioration in neurologic findings, which demands prompt attention.

### Treatment
- **Mild Injuries**: Discharge and observation.
- **Moderate and Severe Injuries**:
  - Optimization of ventilation, oxygenation, and brain perfusion.
  - Treatment of complications such as increased intracranial pressure (ICP), seizures, and hematomas.
  - Surgical intervention may be necessary to place monitors for ICP, decompress the brain, or remove hematomas.
  - Pain relief with short-acting opioids (e.g., fentanyl) as needed.

### Follow-Up
- Many patients require rehabilitation after the initial treatment phase.
- Continuous monitoring for cognitive and neuropsychiatric deficits, which may persist after severe and moderate TBI.

This summary outlines the recommended treatments for individuals with TBI as per the provided context.

**Observations:**
- **Groundedness 5/5, Relevance 5/5.** The answer is fully supported by the retrieved Merck Manual passages and covers the question end to end: CT-based diagnosis, GCS/pupil/vital-sign monitoring, airway and oxygenation, severity-based treatment (mild → observation; moderate/severe → perfusion, ICP, seizures, hematomas, surgery), pain control with fentanyl, and rehabilitation.
- This is the clearest example of RAG's value for **safety**: the base LLM recommended **corticosteroids** for brain injury (a harmful, outdated practice), while the grounded answer contains **no unsupported or unsafe advice**.
- The high relevance score reflects coverage, not depth: numeric targets (BP, CPP, ICP), osmotic therapy (mannitol/hypertonic saline) and specific seizure-prophylaxis drugs aren't included because they weren't in the top-3 chunks.
- Overall this is the most trustworthy answer across all approaches for this question: safe, traceable to the source, and clinically well organised.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [66]:
# Write code to call generate_ground_relevance_response() for the query above and print both scores

# ==============================================================
# CELL E8: EVALUATE QUERY 5 - LEG FRACTURE (k=3 vs k=6)
# k=3 retrieved stress-fracture passages earlier; k=6 fixed that.
# Scoring both shows whether the judge picks up the difference.
# ==============================================================

# --- Evaluate with the default k=3 ---
ground_5, relevance_5, eval_answer_5 = generate_ground_relevance_response(user_input_5)

# --- Evaluate with k=6 (the best setting from fine-tuning) ---
ground_5_k6, relevance_5_k6, eval_answer_5_k6 = generate_ground_relevance_response(user_input_5, k=6)

# --- Compare the scores side by side ---
print("k=3  ->  Groundedness:", ground_5, " | Relevance:", relevance_5)
print("k=6  ->  Groundedness:", ground_5_k6, " | Relevance:", relevance_5_k6)

# --- Show the k=6 answer (the better configuration) ---
print("\n" + "=" * 80)
print("ANSWER THAT WAS SCORED (k=6)")
print("=" * 80)
display(Markdown(eval_answer_5_k6))

k=3  ->  Groundedness: {groundedness_score: 4}  | Relevance: {relevance_score: 5}
k=6  ->  Groundedness: {groundedness_score: 4}  | Relevance: {relevance_score: 5}

ANSWER THAT WAS SCORED (k=6)


### Assessment
- **Injury Type**: Determine if the fracture is open or closed.
- **Symptoms**: Assess for pain, swelling, and any signs of compartment syndrome (e.g., increased tissue pressure, pain on passive stretching).
- **Complications**: Monitor for potential complications such as rapid blood loss, fat embolism, or nerve injuries.

### Management
- **Immediate Care**: 
  - Ensure the patient is moved to a safe location away from potential hazards.
  - Immobilize the leg in a functional position just below heart level.
  - Remove any constrictive clothing or jewelry.

### Treatment
- **Analgesia**: Administer pain relief as needed.
- **Splinting**: Apply a splint to stabilize the fracture.
- **Wound Care**: If the fracture is open, apply sterile dressings and consider tetanus prophylaxis and broad-spectrum antibiotics (e.g., a 2nd-generation cephalosporin plus an aminoglycoside).
- **Transport**: Rapidly transport the patient to a medical facility for further evaluation and treatment.

### Follow-Up
- **Monitoring**: After initial treatment, monitor for complications such as compartment syndrome, deep venous thrombosis, and infection.
- **Rehabilitation**: Early mobilization and physical therapy are essential for recovery, especially in elderly patients or those with coexisting disorders.
- **Long-term Care**: Consider the need for ongoing assessment of functional capacity and potential complications during recovery.

### Considerations for Care and Recovery
- **Age Factors**: Elderly patients may require special attention due to a higher risk of complications from immobilization.
- **Functional Recovery**: The goal is a rapid return to activities of daily living rather than perfect limb alignment.
- **Preventive Measures**: Implement measures to prevent contractures, pressure ulcers, and pneumonia during the recovery phase.

**Observations:**
- **k=3 → Groundedness 4, Relevance 5; k=6 → Groundedness 4, Relevance 5.** This is the only question that didn't get a perfect groundedness score: the model added a few general first-aid points (e.g., moving the patient to a safe location) that aren't explicitly in the retrieved text.
- **The judge missed the k=3 relevance problem.** Earlier, k=3 retrieved *stress-fracture* passages that didn't fit an acute hiking injury, yet relevance still scored 5. The judge checks whether headings "cover" the question, not whether the content fits the clinical scenario. This is a **known limitation of LLM-as-a-judge**.
- The k=6 answer is clearly better on inspection: open vs closed fracture, **compartment syndrome, blood loss, fat embolism and nerve injury**, splinting, open-fracture care (sterile dressing, tetanus prophylaxis, antibiotics), rapid transport, and recovery risks (**DVT**, pressure ulcers, pneumonia, contractures, age-related complications).
- **Scores alone can't separate a good answer from a mis-targeted one**, so automated evaluation should be paired with human spot-checks, especially for safety-critical medical use.

## Actionable Insights and Business Recommendations

**Instructions**: Based on the responses observed across the base LLM, prompt-engineered LLM, RAG, fine-tuning, and evaluation sections above, write down your actionable insights and business recommendations below. Consider aspects such as response completeness (`max_tokens`), groundedness and relevance scores, the effect of retrieval parameters (`k`, chunk size/overlap), and how this solution could be scaled or improved for real-world use.

### Key Insights

1. **RAG made answers safer and traceable.** The base LLM gave fluent but unsourced answers and, in one case, **unsafe advice** (corticosteroids for traumatic brain injury). Grounding in the Merck Manual removed that error, and the judge scored RAG answers **4–5/5 for groundedness** across all five questions.

2. **Prompt engineering improves structure; RAG improves trust.** The system prompt produced clinician-ready structure (assessment → management → treatment → follow-up) and fixed the TBI error, but the model still answered from memory with no source. RAG adds verifiability, which prompting alone can't.

3. **Retrieval quality drives answer quality.** Among all tuning parameters, **`k` had the biggest impact**: raising k from 3 to 6 fixed the leg-fracture answer, which had been built on stress-fracture passages. Temperature, top_p and max_tokens mostly changed wording and length.

4. **Generation settings:** low temperature (0–0.2) gave the most complete, repeatable answers; top_p acted as a guard-rail; **max_tokens below ~400 truncated answers without warning** (the sepsis and hair-loss runs were cut off).

5. **LLM-as-a-judge is useful but not sufficient.** Using a separate, stronger judge model (`gpt-4o`) avoided self-grading bias, but it gave relevance 5/5 to answers containing **neonatal regimens for an adult ICU question** and to the **mis-targeted stress-fracture** answer. Automated scores must be backed by human review.

### Business Recommendations

1. **Deploy as a clinical decision-support tool, not a decision-maker.** Position it as a fast, sourced reference for clinicians, with every answer showing the **Merck Manual page numbers** used and a clear "verify with clinical judgement" notice.

2. **Improve retrieval before scaling:** set **k=5–6** as the default, use MMR search to reduce redundant chunks, and add **metadata filters** (e.g., adult vs paediatric/neonatal) so the wrong patient population isn't retrieved.

3. **Keep the knowledge base current.** Grounded answers inherit the source's content (e.g., dopamine-first vasopressor advice). Schedule regular updates with the latest manual edition and add current guidelines (e.g., Surviving Sepsis Campaign) to the vector store.

4. **Add a human-in-the-loop quality process:** combine automated groundedness/relevance scoring with periodic clinician review of a sample of answers, and track scores over time as a KPI dashboard.

5. **Control cost and latency:** use a small model (`gpt-4o-mini`) for generation and a stronger model only for evaluation, cache frequent questions, and set max_tokens to ~800 to avoid truncated clinical answers.

**Expected impact:** faster access to trusted medical knowledge, fewer errors from unsourced AI answers, and more consistent, standardised care across clinicians.

<font size=6 color='blue'>Power Ahead</font>
___